[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pablosilvoni/Biomimesis/blob/main/Colab/V40_5bases/curva_j_v40_5_colab.ipynb)

**Versión para Google Colab.** Antes: subir la carpeta `Colab/V40_5bases` del repositorio a Google Drive, en `Mi unidad/Biomimesis/Colab/V40_5bases`. Después: *Entorno de ejecución → Ejecutar todas* y aceptar la conexión con Drive. La primera celda toma de ahí las tablas de Masson, las bases, las etiquetas enteras y las redes ya entrenadas: no se entrena nada (≈ 15–25 min en total, casi todo en las Celdas 24, 25 y 27, que resuelven miles de problemas MILP).

# V40 — Red entrenada con láminas enteras (idea del director)

**Idea de V40.** En V39 la red aprende láminas **reales** del MCO y el redondeo se hace al final. En V40, antes de entrenar, cada una de las 50 000 curvas del dataset se redondea con el **minimax (MILP, HiGHS)** y la red aprende directamente esas **láminas enteras** (Celda 7, ≈ 60 min la primera vez; se guardan en `etiquetas_enteras_v40_N.npz`). La Celda 27 compara esta red con la de V39 (entrenada con MCO real), con y sin redondeo óptimo a su salida.

**Qué hace este cuaderno.** Trabaja con la **tensión circunferencial σθθ en r_m** (borde media–adventicia) en función del estiramiento circunferencial λθ, para los 41 sujetos de Masson et al. (2011), entre λθ = 1,02 y 1,16. σθθ sale directo de la ley de Masson con los parámetros de sus Tablas 2 y 4: **no hace falta suponer el espesor de la pared** (en presión había que suponer 0,6 mm).

**Bases.** Cada base es una lámina de un material: A_j(λ) = S_j·max(0, λ − k_j) (recta truncada). Los nudos k_j y las pendientes S_j se ajustaron **desde cero** a las curvas de σ (Celda 3).

**Curvas límite.** y1 (borde blando: ley de Masson con la geometría de NT16, toca el mínimo medido en λ = 1,02) e y2 (borde rígido: HT09 hasta λ = 1,10 y luego llega a 300 kPa). Los 41 sujetos quedan dentro.

**Dos formas de mostrar cada curva** (Celda 23):
- **Valores absolutos:** σθθ [kPa].
- **Adimensional:** (σ − σ_mín)/Δσ, con Δσ = σ(1,16) − σ(1,02) de esa curva: va de 0 a 1 y compara solo la forma. El error en % del rango es el mismo en las dos.

**MCO y red, siempre los dos**, contra la curva original. La red se entrena en kPa; la forma adimensional se calcula después (no hace falta otra red).

**Celdas.**
- **0 a 11:** configuración, curvas σθθ de los 41 sujetos, espacio de trabajo, bases, dataset y red (entrena si no están sus archivos).
- **12 a 18:** evaluación de la red.
- **19:** diseñador (láminas de cada material para una curva σθθ).
- **20 a 22:** láminas aisladas, bases con su fórmula, extrapolación.
- **23:** las dos formas (kPa y adimensional) para y2, Mean HT, y1 y los 41 sujetos.
- **24:** **redondeo a láminas enteras** (no se fabrica media lámina). Compara, contra la curva original: MCO real, red real, redondeo simple de la red, **«Redondeo por mínimos cuadrados enteros (Schnorr–Euchner)»** y **«Redondeo minimax por programación lineal entera mixta (MILP, HiGHS)»**. Tablas y curvas para y2, Mean HT, y1; resumen de los 41 sujetos y de 2000 curvas del espacio de trabajo (`redondeo_v40_N.csv`).
- **25:** robustez: curvas con ruido (1 % y 3 %) y con solo 10 puntos medidos; el minimax se aplica a la curva medida, a la curva del MCO y a la de la red, y el error se mide contra la curva verdadera (`robustez_v40_N.csv`).
- **26:** tiempo de cálculo por curva de cada método.

- **27:** comparación de las dos redes: «Red MCO (V39)» y «Red enteras (V40)». Para cada una: salida real, redondeo simple, Schnorr–Euchner y minimax aplicados a su salida; referencia: minimax sobre la curva original. 41 sujetos, y2, Mean HT, y1 y 2000 curvas del espacio de trabajo (`comparacion_redes_v40_N.csv` y `.png`).

**Archivos de la red:** `modelo_rn_curva_j_v40_N_enteras.keras` (esta red) y, para la Celda 27, la red de V39 (`modelo_rn_curva_j_v35_N.keras`).

**Archivos en esta carpeta:** `tabla2_masson_NT.csv`, `tabla4_masson_HT.csv`, `masson2011_curvas_medias.csv`, la red de V40 (`modelo_rn_curva_j_v40_N_enteras.keras`, `scalers_curva_j_v40_N_enteras.pkl`, `historial_entrenamiento_v40_N_enteras.csv`), la de V39 para comparar (`…_v35_N…`), las etiquetas enteras (`etiquetas_enteras_v40_N.npz`), la plantilla de ensayos y las bases (`bases_v40_N.csv`).

**De arriba hacia abajo.** Nunca hay que volver a una celda anterior. Los nudos y las pendientes de las bases se guardan en **`bases_v40_N.csv`** (columnas material, nudo, pendiente_lamina_kPa): la Celda 3 los lee de ahí y, si el archivo no está (o `OPTIMIZAR_NUDOS = True`), busca los nudos (≈ 25 min) y lo escribe sola. Las curvas de las bases quedan en `funciones_base_v40_N.csv`. La red guarda los nudos y las bases con que se entrenó y avisa si no coinciden.

**Ensayos de los materiales.** Las bases ideales son rectas truncadas (fibras como hilos: no tiran hasta su nudo y después estiran con rigidez constante). Cuando haya ensayos, cada base pasa a ser la **curva medida** de ese material:
1. Un solo archivo, **`ensayos_v40_N.csv`**, con la columna `lambda` (estiramiento circunferencial del injerto, el mismo λθ de las curvas, con el montaje y el rizado ya incluidos) y una columna `material_j` [MPa] por cada material ensayado. La plantilla `plantilla_ensayos_v40_N.csv` muestra el formato (contiene las bases ideales).
2. Ponerlo en la carpeta: el cuaderno detecta solo qué materiales tienen columna; los demás siguen ideales. Conviene medir λ de 1,02 a 1,16 como mínimo.
3. Correr el cuaderno: la Celda 3 informa cuánto se aparta cada ensayo de su recta ideal (y avisa si un ensayo no cubre λ de 1,02 a 1,16) y la Celda 11 **reentrena la red** (≈ 40–60 min) con un nombre propio (`modelo_rn_curva_j_v40_N_ensayos_….keras`). Las veces siguientes la carga.

La curva medida se escala para que en λ = 1,16 valga lo mismo que la recta ideal: una «lámina» conserva su tamaño y cambia solo la forma. Sin ensayos, el cuaderno usa la red `v40_N_enteras`.

**Pendiente:** el paso de fracciones de material a micrómetros de cada capa (diseño del injerto en pared gruesa con estas curvas).

## Preparación — archivos desde GitHub

In [ ]:
# PREPARACIÓN PARA GOOGLE COLAB: busca los datos y las redes entrenadas
#   1) en la carpeta actual; 2) en Google Drive: /content/drive/MyDrive/Biomimesis/Colab/V40_5bases  (subir ahí la carpeta Colab/V40_5bases del repositorio);
#   3) si el repositorio de GitHub es público, los baja de ahí.
import os, sys, subprocess, urllib.parse, urllib.request
import importlib.metadata as _md
if tuple(int(v) for v in _md.version("keras").split(".")[:2]) < (3, 15):    # las redes se guardaron con Keras 3.15
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras>=3.15"], check=True)
    print("Keras actualizado a", _md.version("keras"))
ARCHIVOS = ['tabla2_masson_NT.csv', 'tabla4_masson_HT.csv', 'masson2011_curvas_medias.csv', 'bases_v40_5.csv', 'etiquetas_enteras_v40_5.npz', 'modelo_rn_curva_j_v40_5_enteras.keras', 'scalers_curva_j_v40_5_enteras.pkl', 'historial_entrenamiento_v40_5_enteras.csv', 'modelo_rn_curva_j_v35_5.keras', 'scalers_curva_j_v35_5.pkl', 'historial_entrenamiento_v35_5.csv']
CARPETA_DRIVE = "/content/drive/MyDrive/Biomimesis/Colab/V40_5bases"
if any(not os.path.exists(f) for f in ARCHIVOS):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(CARPETA_DRIVE):
            os.chdir(CARPETA_DRIVE)                                  # los resultados también quedan en esa carpeta de Drive
    except ImportError:
        pass
URL = "https://raw.githubusercontent.com/pablosilvoni/Biomimesis/main/Colab/V40_5bases/"
for f in ARCHIVOS:
    if not os.path.exists(f):
        try:
            urllib.request.urlretrieve(URL + urllib.parse.quote(f), f)
            print(f"⬇️  {f} (GitHub)")
        except Exception:
            raise FileNotFoundError(f"Falta {f}: subí la carpeta Colab/V40_5bases del repositorio a Google Drive, en {CARPETA_DRIVE}")
print(f"✅ {len(ARCHIVOS)} archivos listos en {os.getcwd()}")

## Celda 0 — Configuración (ejecutar siempre primero)

In [ ]:
# CELDA 0: CONFIGURACIÓN (ejecutar siempre primero)
import os
import numpy as np

N_BASES = 5                      # ← número de bases (materiales): 4, 5 o 6. Cada número de bases tiene su propia red
VERSION = f"v40_{N_BASES}"
SEED = 42
DIR_MASSON = "."                 # tablas de Masson en la misma carpeta que el notebook
ENTRENAR = "auto"                # "auto": carga la red si sus archivos están en la carpeta y, si no, la entrena;
                                 # True: entrena siempre; False: carga siempre (error si faltan los archivos)
# ENSAYOS DE LOS MATERIALES: un solo CSV con la columna lambda (estiramiento circunferencial del injerto, el mismo λθ de las curvas,
#   con el montaje y el rizado de las fibras ya incluidos) y una columna material_j por cada material ensayado (tensión de Cauchy
#   [MPa]). Los materiales sin columna (o con la columna vacía) siguen con su base ideal (recta truncada: fibra como hilo).
#   Con ensayos, la red se REENTRENA (Celda 11, ≈ 40–60 min). Formato: ver plantilla_ensayos_v40_N.csv.
ARCHIVO_ENSAYOS = f"ensayos_v40_{N_BASES}.csv"   # si el archivo no está en la carpeta: bases ideales
GUARDAR_PLANTILLA_ENSAYO = True  # True: la Celda 3 escribe plantilla_ensayos_v40_N.csv (las bases ideales en el formato del ensayo)
ENSAYOS = []                     # materiales ensayados (se detectan solos)
if os.path.exists(ARCHIVO_ENSAYOS):
    import pandas as pd
    _e = pd.read_csv(ARCHIVO_ENSAYOS)
    ENSAYOS = [j for j in range(1, N_BASES + 1) if f"material_{j}" in _e and _e[f"material_{j}"].notna().sum() >= 2]
# Red: con bases ideales se usa la red de V35 (mismas bases); con ensayos, una red propia que se entrena la primera vez.
# V40 (idea del director): la red se entrena con las láminas ENTERAS del redondeo minimax (MILP, HiGHS) de cada curva, no con las
#   láminas reales del MCO. La red de V39/V35 (etiquetas MCO) se usa solo para comparar (Celda 27).
MODELO_RED = f"{VERSION}_enteras" if not ENSAYOS else f"{VERSION}_enteras_ensayos_" + "_".join(map(str, ENSAYOS))
MODELO_COMPARACION = f"v35_{N_BASES}"   # red entrenada con láminas reales del MCO (V35 = V39), para la Celda 27
ARCHIVO_ETIQUETAS_ENTERAS = f"etiquetas_enteras_{VERSION}.npz"   # minimax de las 50 000 curvas (lento: se guarda y se reutiliza)
if ENTRENAR == "auto":
    ENTRENAR = not os.path.exists(f"modelo_rn_curva_j_{MODELO_RED}.keras")

# V35: las curvas son TENSIONES σθθ(λθ) en r_m (ley de Masson), no presiones. Cada base = UNA lámina de un material:
#   A_j(λ) = S_j·max(0, λ − k_j)   (recta truncada), con S_j = PEND_MAX_j / C_LAM_j.
#   Nudos k_j y pendientes S_j se ajustan desde cero a las curvas de σ (Celda 3).
ARCHIVO_BASES = f"bases_v40_{N_BASES}.csv"   # nudos y pendientes de las bases: la Celda 3 los lee de acá; si no existe, busca los
                                             # nudos (≈ 25 min) y lo escribe (columnas material, nudo, pendiente_lamina_kPa)
OPTIMIZAR_NUDOS = False          # True: vuelve a buscar los nudos (y sobrescribe ARCHIVO_BASES) aunque el archivo exista
NUDOS_FORZADOS = None            # opcional: una lista de N_BASES nudos para imponerlos a mano (ignora el archivo)
UMBRAL_COND = 30.0               # cota de cond(A) (columnas normalizadas) que deben cumplir los nudos
PESO_NEG = 10.0                  # peso de la penalización por láminas negativas del MCO libre al elegir los nudos (evaluada en todo el espacio)
SIGNO_ESTRICTO = True            # True: la Celda 3 exige 0 curvas del espacio de trabajo con láminas < −TOL_NEG
SEP_MIN_NUDOS = 0.008            # separación mínima entre nudos
MARGEN_NUDO_FINAL = 0.012        # el último nudo queda ≥ 0,012 antes de X_MAX: si no, su material actúa en muy pocos puntos y la red no lo distingue
TOL_NEG = 0.01                   # lámina negativa tolerada (en láminas de la red, de 10): 0,1 % del máximo
C_LAM = 20                       # máximo de láminas por material (configuración adoptada: 20) (solo para las Celdas 3 a 22): un número para todos o una lista de N_BASES valores; no requiere reentrenar
C_LAM_RED = 10.0                 # láminas con que se entrena la red (no cambiar)

ERROR_MAX_PCT = 3.0              # criterio: error punto a punto ≤ 3 % del rango de la curva
SUJETO_Y1, SUJETO_Y2 = "NT16", "HT09"                            # sujetos de partida de las curvas límite
# y1: ley de Masson con la geometría y las fibras de NT16 y estos 5 parámetros (dentro del rango de los 41 sujetos),
#     elegidos para que y1(X_MIN) = mín. de los 41 sujetos en X_MIN (HT10, 4,71 kPa) y y1 ≤ los 41 sujetos en todo el dominio
PARAM_Y1 = {"c_kPa": 43.6135, "c1_kPa": 5.0302, "c1circ_kPa": 5.1034, "c2circ": 11.8183, "Tm_kPa": 22.3336}
LAMBDA_Y2, Y2_MAX = 1.10, 300.0  # y2 = HT09 hasta LAMBDA_Y2 y luego llega a Y2_MAX [kPa] en X_MAX (≥ los 41 sujetos)
X_MIN, X_MAX, N_PTS = 1.02, 1.16, 999
X = np.linspace(X_MIN, X_MAX, N_PTS)
FRAC_SUJETOS_TEST = 0.25
X_INICIO_CONVEXIDAD, TOL_FORMA_REL = 1.05, 0.01                  # control de forma J
TOL_CLASE_NRMSE, N_NUDOS_CLASE, NUDO_MIN_CLASE = 2.0, 40, 0.85   # control de clase a la entrada
ETQ_X, ETQ_Y = r"$\lambda_\theta$ (en $r_m$)", r"$\sigma_{\theta\theta}$ en $r_m$ [kPa]"
ETQ_YN = r"$(\sigma - \sigma_{mín})/\Delta\sigma$"     # forma adimensional: 0 en λ = X_MIN, 1 en λ = X_MAX

_v = MODELO_RED
ARCHIVO_MODELO, ARCHIVO_SCALERS, ARCHIVO_HISTORIAL = (f"modelo_rn_curva_j_{_v}.keras", f"scalers_curva_j_{_v}.pkl",
                                                      f"historial_entrenamiento_{_v}.csv")
print(f"✅ {VERSION} | curvas σθθ(λθ) en r_m | λθ ∈ [{X_MIN}, {X_MAX}] | {N_BASES} bases "
      f"({'ideales' if not ENSAYOS else 'ensayos de los materiales ' + str(ENSAYOS)}) | "
      f"red: {'se entrena → ' if ENTRENAR else 'se carga de '}{ARCHIVO_MODELO}")

## Celda 1 — Modelo de Masson et al. (2011), curvas σθθ de los 41 sujetos y curvas límite

In [ ]:
# CELDA 1: MODELO DE MASSON ET AL. (2011) Y CURVAS σθθ(λθ) DE LOS 41 SUJETOS
# σθθ: tensión circunferencial de Cauchy en r_m (borde media–adventicia), como en las Figs. 5a y 7a de Masson.
# Sale directo de la ley de Masson con los parámetros de las Tablas 2 y 4: no hace falta suponer el espesor de la pared.
import os
import numpy as np
import pandas as pd

COLS_PARAM = ["Rm_mm", "Theta0_deg", "lambda_ax", "c_kPa", "c1_kPa", "c2", "c1circ_kPa", "c2circ", "alpha_deg",
              "Tm_kPa", "lambda0", "lambda_m", "a_kPa", "b"]


def sigma_masson(lt, p, restar_Pa=True):
    """σθθ(λθ) [kPa] en r_m: elastina neo-Hookeana + 4 familias de colágeno + tono activo."""
    lt = np.asarray(lt, float)
    lz, al = p["lambda_ax"], np.deg2rad(p["alpha_deg"])
    lam_d = np.sqrt(lt ** 2 * np.sin(al) ** 2 + lz ** 2 * np.cos(al) ** 2)
    suma = np.zeros_like(lt)
    for lam_k, c1, c2, ang in [(lz * np.ones_like(lt), p["c1_kPa"], p["c2"], 0.0), (lt, p["c1circ_kPa"], p["c2circ"], np.pi / 2),
                               (lam_d, p["c1_kPa"], p["c2"], al), (lam_d, p["c1_kPa"], p["c2"], -al)]:
        E = lam_k ** 2 - 1.0
        suma = suma + c1 * E * np.exp(c2 * E ** 2) * np.sin(ang) ** 2
    activo = p["Tm_kPa"] * lt * (1.0 - ((p["lambda_m"] - lt) / (p["lambda_m"] - p["lambda0"])) ** 2)
    s = -p["c_kPa"] * (1.0 / (lt * lz)) ** 2 + p["c_kPa"] * lt ** 2 + lt ** 2 * suma + activo
    return s - p["a_kPa"] * np.exp(p["b"]) if restar_Pa else s


def curva_objetivo(lt_m, p):
    """σθθ(λθ) en r_m [kPa] (ley de Masson, con la presión perivascular P_a restada como en el paper)."""
    return sigma_masson(np.asarray(lt_m, float), p, restar_Pa=True)


def diagnostico_forma(y, x=X):
    """(monótona, convexa desde X_INICIO_CONVEXIDAD), con tolerancia relativa a la pendiente máxima."""
    p = np.diff(y) / np.diff(x)
    tol = TOL_FORMA_REL * max(float(np.abs(p).max()), 1e-9)
    return bool(p.min() >= -tol), bool(np.diff(p)[x[1:-1] >= X_INICIO_CONVEXIDAD].min() >= -tol)


POB = pd.concat([pd.read_csv(f"{DIR_MASSON}/{f}").assign(grupo=g) for f, g in
                 [("tabla2_masson_NT.csv", "NT"), ("tabla4_masson_HT.csv", "HT")]], ignore_index=True)
POB["id"] = POB["grupo"] + POB["subject"].map("{:02d}".format)
POB_Y = np.array([curva_objetivo(X, r) for _, r in POB.iterrows()])

# Sujetos reservados para test (estratificado por grupo): no entran al generador
_rng = np.random.default_rng(SEED)
POB["test"] = False
for g in ("NT", "HT"):
    i_g = POB.index[POB["grupo"] == g].to_numpy()
    POB.loc[_rng.choice(i_g, size=int(round(FRAC_SUJETOS_TEST * len(i_g))), replace=False), "test"] = True
MASK_TEST = POB["test"].to_numpy()
MASK_TRAIN = ~MASK_TEST

# Curvas límite: y1 = ley de Masson (geometría de NT16 con PARAM_Y1), por debajo de los 41 sujetos y con y1(X_MIN) = mín. medido;
# y2 = HT09 + a·(exp(b·E²) − 1), E = (λ/LAMBDA_Y2)² − 1 > 0, con b = c2circ de HT09 y a tal que y2(X_MAX) = Y2_MAX
_p1 = POB.loc[POB["id"] == SUJETO_Y1].iloc[0].copy()
for _k, _v in PARAM_Y1.items():
    _p1[_k] = _v
Y1 = curva_objetivo(X, _p1)
_y2 = POB_Y[(POB["id"] == SUJETO_Y2).to_numpy()][0]
_g = np.exp(float(POB.loc[POB["id"] == SUJETO_Y2, "c2circ"].iloc[0]) * np.maximum(0.0, (X / LAMBDA_Y2) ** 2 - 1.0) ** 2) - 1.0
Y2 = _y2 + (Y2_MAX - _y2[-1]) / _g[-1] * _g
_i0 = int(POB_Y[:, 0].argmin())
assert abs(Y1[0] - POB_Y[_i0, 0]) < 1e-3, "y1(X_MIN) debe ser la menor tensión medida en X_MIN"
assert np.all(Y1 <= POB_Y.min(axis=0) + 1e-6), "y1 debe quedar por debajo de los 41 sujetos"
assert all(diagnostico_forma(Y1)), "y1 debe tener forma J"
ETQ_BANDA = f"Curvas límite y1 (Masson, y1({X_MIN}) = {Y1[0]:.2f} kPa = {POB.id[_i0]}) – y2 ({SUJETO_Y2} → {Y2_MAX:g} kPa)"
print(f"y1: ley de Masson, y1({X_MIN}) = {Y1[0]:.3f} kPa (= {POB.id[_i0]}), y1({X_MAX}) = {Y1[-1]:.2f} kPa; "
      f"distancia mínima a los 41 sujetos fuera de X_MIN: {(POB_Y.min(axis=0) - Y1)[1:].min():.4f} kPa")


def dentro_espacio(Y, tol=1e-9):
    Y = np.atleast_2d(Y)
    return np.all((Y >= Y1 - tol) & (Y <= Y2 + tol), axis=1)


def banda(ax, x=X):
    ax.fill_between(x, np.interp(x, X, Y1), np.interp(x, X, Y2), color="#e5e7eb", alpha=0.6, lw=0, label=ETQ_BANDA)


GRUPOS_MEDIAS = {"Mean Young": (POB["grupo"] == "NT") & (POB["age"] <= 42), "Mean Old": (POB["grupo"] == "NT") & (POB["age"] >= 53),
                 "Mean NT": (POB["grupo"] == "NT") & (POB["age"] >= 39), "Mean HT": POB["grupo"] == "HT"}
MEDIAS = {k: POB_Y[m.to_numpy()].mean(axis=0) for k, m in GRUPOS_MEDIAS.items()}

_dent = dentro_espacio(POB_Y)
print(f"✅ {len(POB)} sujetos ({(POB.grupo == 'NT').sum()} NT, {(POB.grupo == 'HT').sum()} HT) | test: {', '.join(POB.id[MASK_TEST])}")
print(f"   σθθ en λ = {X_MIN}: [{POB_Y[:, 0].min():.1f}, {POB_Y[:, 0].max():.1f}] kPa | en λ = {X_MAX}: "
      f"[{POB_Y[:, -1].min():.1f}, {POB_Y[:, -1].max():.1f}] kPa | forma J (monótona y convexa): "
      f"{np.mean([all(diagnostico_forma(y)) for y in POB_Y]):.0%}")
print(f"   Dentro de y1–y2 en todo el dominio: {_dent.sum()}/{len(POB)} | fuera: {', '.join(POB.id[~_dent])}")

# Validación contra las medias de σθθ digitalizadas del paper (Figs. 5a y 7a)
if os.path.exists(f"{DIR_MASSON}/masson2011_curvas_medias.csv"):
    dig = pd.read_csv(f"{DIR_MASSON}/masson2011_curvas_medias.csv")
    for k, m in GRUPOS_MEDIAS.items():
        d = dig[dig.curva == k]
        s = np.mean([sigma_masson(d.lambda_theta.to_numpy(), r) for _, r in POB[m].iterrows()], axis=0)
        e = np.sqrt(np.mean((s - d.sigma_theta_kPa.to_numpy()) ** 2))
        print(f"   σθθ reconstruida vs. paper, {k:<10} (n = {m.sum():>2}): RMSE {e:.2f} kPa ({e / d.sigma_theta_kPa.mean():.1%})")
pd.DataFrame(POB_Y.T, index=pd.Index(np.round(X, 6), name="lambda_theta"), columns=POB["id"]).to_csv(
    f"curvas_tension_poblacion_{VERSION}.csv")

## Celda 2 — Espacio de trabajo: curvas fisiológicas sintéticas

In [ ]:
# CELDA 2: ESPACIO DE TRABAJO — CURVAS FISIOLÓGICAS SINTÉTICAS (no dependen de las bases)
#   2 constitutiva / 3 amplia / 5 uniforme: ley de Masson dentro de y1–y2   4 exponencial: otra ley   6 pacientes de entrenamiento
#   Con las familias 2, 3 y 5 (el espacio de trabajo) la Celda 3 calcula las pendientes máximas y, si hace falta, los nudos.
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

N_SAMPLES = 50000
FRAC = {0: 0.20, 1: 0.20, 2: 0.20, 4: 0.10, 6: 0.05, 5: 0.10}      # la amplia (3) completa el total
FAMILIAS = {0: "pura", 1: "combinada", 2: "constitutiva", 3: "constitutiva_amplia", 4: "exponencial",
            5: "constitutiva_uniforme", 6: "pacientes"}
FAMILIAS_OBJETIVO = (2, 3, 4, 5, 6)
Y_TECHO, RUIDO_CONST = 1.5 * float(POB_Y.max()), 0.01
PERTURB = {2: dict(k=(2, 3), sd_mult=0.10, sd_adit=0.15, amp=(0.85, 1.20), gamma=(1.0, 1.0)),
           3: dict(k=(2, 4), sd_mult=0.20, sd_adit=0.30, amp=(0.65, 1.50), gamma=(1.0, 1.35)),
           4: dict(k=(2, 3), sd_log=0.15, amp=(0.70, 1.40))}
PARAM_MULT = ["Rm_mm", "c_kPa", "c1_kPa", "c1circ_kPa", "Tm_kPa", "a_kPa"]          # perturbación log-normal
PARAM_ADIT = ["Theta0_deg", "lambda_ax", "c2", "c2circ", "alpha_deg", "lambda0", "lambda_m", "b"]   # en desvíos poblacionales
rng = np.random.default_rng(2026)
N_FAM = {f: int(round(v * N_SAMPLES)) for f, v in FRAC.items()}
N_FAM[3] = N_SAMPLES - sum(N_FAM.values())
ruido = lambda Y, r, g: np.maximum(Y + g.normal(0.0, r, size=Y.shape) * np.abs(Y), 0.0)
LIMPIAS, RUIDOSAS, ACEPTACION = {}, {}, {}

# C) leyes constitutivas: mezcla convexa (Dirichlet) de sujetos de entrenamiento + perturbación + amplitud
TH = POB.loc[MASK_TRAIN, COLS_PARAM].to_numpy(float)
SD = POB[COLS_PARAM].to_numpy(float).std(axis=0)
ix = {c: i for i, c in enumerate(COLS_PARAM)}
iM, iA = [ix[c] for c in PARAM_MULT], [ix[c] for c in PARAM_ADIT]
es_mult = np.isin(np.arange(len(COLS_PARAM)), iM)
a_z = lambda t: np.where(es_mult, np.log(np.maximum(t, 1e-9)), t)       # módulos en escala log
de_z = lambda z: np.where(es_mult, np.exp(np.where(es_mult, z, 0.0)), z)
Z = a_z(TH)
amplitud = lambda cfg: float(np.exp(rng.uniform(np.log(cfg["amp"][0]), np.log(cfg["amp"][1]))))


def ley_exp(x, C, mu, k1, k2):
    E = x ** 2 - 1.0
    return C + mu * E + k1 * E * np.exp(k2 * E ** 2)


def valida(y, acotar=True):
    return (np.all(np.isfinite(y)) and y.min() > 0 and y.max() <= Y_TECHO and all(diagnostico_forma(y))
            and (not acotar or bool(dentro_espacio(y)[0])))


def masson(th):
    return None if th[ix["lambda_m"]] <= th[ix["lambda0"]] + 0.2 else curva_objetivo(X, dict(zip(COLS_PARAM, th)))


def sortear(f):
    """Una curva de la familia f, o None si se rechaza."""
    if f == 5:                                                      # uniforme: parámetros independientes en [mín, máx]
        y = masson(de_z(rng.uniform(a_z(TH.min(0)), a_z(TH.max(0)))))
        return y if y is not None and valida(y) else None
    cfg = PERTURB[f]
    k = rng.integers(cfg["k"][0], cfg["k"][1] + 1)
    idx = rng.choice(len(TH), size=k, replace=False)
    w = rng.dirichlet(np.ones(k))
    if f == 4:                                                      # exponencial: media geométrica de ajustes
        p = np.exp(w @ np.log(P_EXP[idx]) + rng.normal(0.0, cfg["sd_log"], size=4))
        y = amplitud(cfg) * ley_exp(X, *p)
        return y if valida(y, acotar=False) else None
    z = Z.mean(0) + rng.uniform(*cfg["gamma"]) * (w @ Z[idx] - Z.mean(0))     # γ > 1: extrapola desde el centro
    z[iM] += rng.normal(0.0, cfg["sd_mult"], size=len(iM))
    z[iA] += rng.normal(0.0, cfg["sd_adit"], size=len(iA)) * SD[iA]
    th = de_z(z)
    th[ix["alpha_deg"]] = np.clip(th[ix["alpha_deg"]], 5.0, 85.0)
    th[ix["lambda_ax"]] = max(th[ix["lambda_ax"]], 1.0)
    th[[ix["c2"], ix["c2circ"]]] = np.maximum(th[[ix["c2"], ix["c2circ"]]], 0.5)
    th[ix["Theta0_deg"]] = np.clip(th[ix["Theta0_deg"]], 60.0, 300.0)
    y = masson(th)
    if y is None:
        return None
    y = amplitud(cfg) * y
    return y if valida(y) else None


P_EXP = np.maximum([curve_fit(ley_exp, X, y, p0=(max(y[0] - 5.0, 1.0), 50.0, 20.0, 10.0), maxfev=40000,
                              bounds=([0, 0, 1e-3, 1e-3], [1e3, 2e3, 2e3, 200.0]))[0] for y in POB_Y[MASK_TRAIN]], 1e-3)
for f in (2, 3, 4, 5):
    Y, intentos = [], 0
    while len(Y) < N_FAM[f]:
        y, intentos = sortear(f), intentos + 1
        if y is not None:
            Y.append(y)
    LIMPIAS[f], ACEPTACION[f] = np.array(Y), N_FAM[f] / intentos
    RUIDOSAS[f] = ruido(LIMPIAS[f], RUIDO_CONST, rng)
LIMPIAS[6] = POB_Y[MASK_TRAIN][np.resize(np.arange(MASK_TRAIN.sum()), N_FAM[6])]       # pacientes reales (sin recorte en 0)
RUIDOSAS[6] = LIMPIAS[6] + rng.normal(0.0, RUIDO_CONST, size=LIMPIAS[6].shape) * np.abs(LIMPIAS[6])

W_ESP = np.vstack([LIMPIAS[f] for f in (2, 3, 5)])                         # espacio de trabajo (curvas de Masson en y1–y2)
print(f"✅ Espacio de trabajo: {len(W_ESP)} curvas de Masson dentro de y1–y2 | aceptación: "
      + ", ".join(f"{FAMILIAS[f]} {100 * ACEPTACION[f]:.1f}%" for f in (2, 3, 4, 5)) + f" | σθθ en λ = {X_MAX}: "
      f"p1 {np.percentile(W_ESP[:, -1], 1):.1f} | p50 {np.median(W_ESP[:, -1]):.1f} | máx. {W_ESP[:, -1].max():.1f} kPa")

## Celda 3 — Bases: nudos, pendientes y ensayos de los materiales

### Documentación de la Celda 3: cómo se construyen las bases (métodos, para el artículo)

**Qué es una base.** Una base es **una lámina de un material**: A_j(λ) = S_j · max(0, λ − k_j), con k_j el **nudo** (estiramiento en que el material empieza a tirar) y S_j la **pendiente de una lámina** [kPa]. La curva de cualquier arteria se aproxima como σ(λ) ≈ Σ c_j·A_j(λ), donde c_j es la cantidad de láminas del material j.

**Funciones de la celda**

| Función | Qué hace |
|---|---|
| `forma_base(lam, k)` | Forma de cada base: recta truncada max(0, λ − k_j) o, si el material tiene ensayo, su curva medida (escalada). |
| `leer_ensayo(archivo, j)` | σ(λ) del material j desde `ensayos_v38_N.csv`: 0 antes del primer punto medido, interpolación lineal entre puntos, recta con la última pendiente después del último. |
| `lineas(k)` | Matriz de bases sin escalar (999 puntos de λ × N materiales), con los nudos ordenados. |
| `cond_A(k)` | Número de condición de la matriz de bases con columnas normalizadas a máximo 1 (cociente entre el mayor y el menor valor singular). |
| `optimizar_nudos(n_b)` | Búsqueda de los nudos óptimos (ver método abajo). Solo corre si falta `bases_v38_N.csv`, si `OPTIMIZAR_NUDOS = True` o si cond(A) ≥ UMBRAL_COND. |
| `n_negativas(k)` | Cuenta curvas del espacio de trabajo y sujetos que necesitarían alguna lámina negativa. |
| `mco(Y, B)` | Láminas de una curva por mínimos cuadrados ordinarios (MCO). |
| `err_pct(Y, B)` | Error máximo punto a punto y NRMSE, en % del rango de cada curva. |

**Métodos**

1. **Nudos óptimos.** Optimización global sin derivadas por **evolución diferencial** (Storn y Price, 1997; `scipy.optimize.differential_evolution`, semilla 42, hasta 300 generaciones, tolerancia 10⁻⁶) seguida de un refinamiento local con el **método simplex de Nelder–Mead** (Nelder y Mead, 1965; hasta 6000 iteraciones). Para cada juego de nudos k se calcula el MCO de 2500 curvas del espacio de trabajo, los sujetos de entrenamiento dentro de y1–y2 e y2 (cada 2 puntos de λ), y se minimiza:
   - la media del peor 1 % del error punto a punto [% del rango] (criterio minimax),
   - \+ 0,25 × el NRMSE medio,
   - \+ 50 × max(0, cond(A) − (UMBRAL_COND − 1)): penaliza bases mal condicionadas,
   - \+ PESO_NEG × 100 × media de las láminas negativas (y 100 × la peor), evaluadas en las 22 500 curvas del espacio, los sujetos de entrenamiento, y1 e y2: se buscan nudos con los que el ajuste libre dé c ≥ 0.

   Restricciones: k_1 entre 0,6 y 0,98; los demás entre 1,0 y X_MAX − 0,012 (el último material debe actuar en al menos 0,012 de λ, si no la red no lo distingue); separación entre nudos ≥ 0,008. Solo se usan sujetos de entrenamiento (los 10 de test no intervienen).
2. **Número de condición.** cond(A) = σ_máx/σ_mín de la matriz de bases con columnas normalizadas, por descomposición en valores singulares (SVD). Umbral: cond(A) < 30.
3. **Pendiente máxima PEND_MAX_j.** Con las rectas sin escalar G, las pendientes que exige cada curva son P·y, con P = G⁺ la **pseudoinversa de Moore–Penrose** (mínimos cuadrados). PEND_MAX_j es la mayor pendiente que piden al material j las 22 500 curvas del espacio, los sujetos dentro de y1–y2 e y2 (redondeada a 0,1 kPa).
4. **Pendiente de una lámina.** S_j = PEND_MAX_j / C_LAM, con C_LAM = 20: **ninguna curva del espacio necesita más de 20 láminas de un material**. La red trabaja con C_LAM_RED = 10 (bases A_RED); láminas del diseño = láminas de la red × 2.
5. **Control de signo.** Ninguna curva del espacio ni ningún sujeto puede requerir una lámina menor que −TOL_NEG = −0,01 (de 10).
6. **MCO.** c = A⁺y por mínimos cuadrados (`numpy.linalg.lstsq`, basado en SVD).
7. **Bases con ensayo.** La curva medida se interpola linealmente y se **escala** para que en λ = X_MAX valga lo mismo que la recta ideal (la lámina conserva su tamaño; cambia la forma). Se informa el desvío RMS entre la forma medida y la recta ideal, en % del rango de la recta, entre X_MIN y X_MAX.

**Ejemplo a mano: material 4 con 6 materiales**
- Nudo k_4 = 1,1099; PEND_MAX_4 = 1155,7 kPa.
- Una lámina: S_4 = 1155,7 / 20 = **57,79 kPa**.
- En λ = 1,16: A_4 = 57,79 × (1,16 − 1,1099) = 57,79 × 0,0501 = **2,90 kPa por lámina**. Antes de λ = 1,1099 no aporta nada.

**Archivo `bases_v38_6.csv`** (lo escribe esta celda cuando busca los nudos):
```
material,nudo,pendiente_lamina_kPa
1,0.8193,12.315
2,1.0227,41.39
3,1.0774,34.73
4,1.1099,57.785
5,1.1318,83.23
6,1.1478,111.13
```

**Referencias de los métodos**
- Storn, R., Price, K. (1997). Differential evolution: a simple and efficient heuristic for global optimization over continuous spaces. *J. Global Optim.* 11, 341–359.
- Nelder, J. A., Mead, R. (1965). A simplex method for function minimization. *Comput. J.* 7(4), 308–313.
- Penrose, R. (1955). A generalized inverse for matrices. *Proc. Camb. Phil. Soc.* 51(3), 406–413.
- Golub, G. H., Van Loan, C. F. (2013). *Matrix Computations*, 4.ª ed. Johns Hopkins (SVD, número de condición, mínimos cuadrados).
- Virtanen, P. et al. (2020). SciPy 1.0. *Nat. Methods* 17, 261–272. Harris, C. R. et al. (2020). Array programming with NumPy. *Nature* 585, 357–362.

In [ ]:
# CELDA 3: BASES = UNA LÁMINA DE CADA MATERIAL, A_j(λ) = S_j·max(0, λ − k_j), S_j = PEND_MAX_j / C_LAM_j
#   Nudos y pendientes se ajustan desde cero a las curvas de σθθ. (Se probó la forma λ²·max(0, λ − k): el MCO ajusta igual,
#   pero la red aprende 2 veces peor; por eso se usan rectas truncadas.)
# 1) Nudos: se leen de ARCHIVO_BASES (Celda 0). Si el archivo no existe, si OPTIMIZAR_NUDOS = True o si no cumplen cond(A) < UMBRAL_COND,
#    se buscan y se GUARDAN en ese archivo (no hay que copiar nada a la Celda 0). Búsqueda: minimax (media del peor 1 % del error punto a punto del MCO en el espacio de trabajo) con cond(A) < UMBRAL_COND
#    y penalización PESO_NEG por láminas negativas del MCO libre en TODO el espacio de trabajo (la base se elige para que el
#    ajuste sin condiciones dé c ≥ 0; las pendientes no cambian el signo: solo escalan cada columna).
#    Búsqueda: evolución diferencial (global, sin derivadas) y refinamiento final con Nelder–Mead.
# 2) PEND_MAX = máxima pendiente que exige el espacio de trabajo a cada capa (MCO con líneas sin escalar).
# Cambiar C_LAM solo cambia la rigidez de cada lámina, no el ajuste: c_j (diseño) = c_j (red) · C_LAM_j / C_LAM_RED.
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution, minimize


FORMA_ENSAYO = {}                                     # material (0 a N−1) → forma medida (se llena abajo si hay ENSAYOS)


def forma_base(lam, k):
    """Forma de cada base (una columna por nudo): recta truncada max(0, λ − k) o, si el material tiene ensayo, su curva medida."""
    lam = np.asarray(lam, float)[:, None]
    F = np.maximum(0.0, lam - np.asarray(k, float))
    for j, f in FORMA_ENSAYO.items():
        F[:, j] = f(lam[:, 0])
    return F


def leer_ensayo(archivo, j):
    """σ [kPa] del material j según el CSV de ensayos (columnas lambda y material_j): interpolación lineal, 0 antes del primer
    punto medido y recta con la última pendiente después del último."""
    d = pd.read_csv(archivo)[["lambda", f"material_{j}"]].dropna().sort_values("lambda")
    le, se = d["lambda"].to_numpy(float), d[f"material_{j}"].to_numpy(float) * 1e3
    m = (se[-1] - se[-2]) / (le[-1] - le[-2])
    return lambda l: np.where(np.asarray(l) <= le[-1], np.interp(l, le, se, left=0.0), se[-1] + m * (np.asarray(l) - le[-1]))


def lineas(k, x=X):
    return forma_base(x, np.sort(np.asarray(k, float)))


def cond_A(k):
    s = np.linalg.svd(lineas(k) / lineas(k).max(axis=0), compute_uv=False)
    return s[0] / s[-1]


def optimizar_nudos(n_b):
    """Nudos minimax para n_b bases: cond(A) < UMBRAL_COND y láminas ≥ 0 (evolución diferencial + refinamiento Nelder–Mead)."""
    W = np.vstack([W_ESP[np.random.default_rng(1).choice(len(W_ESP), min(2500, len(W_ESP)), replace=False)],
                   POB_Y[MASK_TRAIN & dentro_espacio(POB_Y)], Y2])[:, ::2]     # solo datos de entrenamiento
    xs, rg = X[::2], np.ptp(W, axis=1)
    WN = np.vstack([W_ESP, POB_Y[MASK_TRAIN], Y1, Y2])             # signo: todo el espacio + sujetos de entrenamiento + curvas límite (999 puntos)

    def obj(k):
        k = np.sort(k)
        if np.diff(k).min() < SEP_MIN_NUDOS or k[-1] > X_MAX - MARGEN_NUDO_FINAL or k[0] < 0.6:
            return 1e9                                           # fuera de las restricciones: peor que cualquier punto permitido
        G_ = lineas(k, xs)
        Pk = np.linalg.pinv(G_)
        C = Pk @ W.T                                                   # pendientes de cada capa [kPa], una columna por curva
        R = W.T - G_ @ C
        e = np.sort(np.abs(R).max(axis=0) / rg * 100)
        CN = np.linalg.pinv(lineas(k)) @ WN.T
        lam = C_LAM_RED * CN / np.maximum(CN.max(axis=1, keepdims=True), 1e-9)   # láminas (escala de la red)
        return e[-len(e) // 100:].mean() + 0.25 * np.mean(np.sqrt(np.mean(R ** 2, axis=0)) / rg * 100) \
            + 50.0 * max(0.0, cond_A(k) - (UMBRAL_COND - 1.0)) + PESO_NEG * 100 * (neg := np.maximum(0.0, -lam).sum(axis=0)).mean() \
            + (100.0 * neg.max() if SIGNO_ESTRICTO else 0.0)            # estricto: castiga la curva más negativa
    rango = [(0.6, 0.98)] + [(1.0, X_MAX - MARGEN_NUDO_FINAL)] * (n_b - 1)   # k_1 antes del dominio; los demás dentro
    r = differential_evolution(obj, rango, seed=SEED, tol=1e-6, maxiter=300, polish=False)
    r = minimize(obj, r.x, method="Nelder-Mead", options=dict(maxiter=6000, xatol=1e-5, fatol=1e-6))
    return np.round(np.sort(r.x), 4)


def n_negativas(k):
    """Curvas del espacio de trabajo y sujetos con alguna lámina MCO < 0 (el signo no depende de las pendientes)."""
    P_ = np.linalg.pinv(lineas(k))
    return int(((P_ @ W_ESP.T) < 0).any(axis=0).sum()), int(((P_ @ POB_Y.T) < 0).any(axis=0).sum())


_nuevos = False
if NUDOS_FORZADOS is not None:
    NUDOS, _origen = np.sort(np.asarray(NUDOS_FORZADOS, float)), "NUDOS_FORZADOS (Celda 0)"
elif os.path.exists(ARCHIVO_BASES):
    NUDOS, _origen = pd.read_csv(ARCHIVO_BASES).sort_values("material")["nudo"].to_numpy(float), f"'{ARCHIVO_BASES}'"
else:
    NUDOS, _origen = np.array([]), "ninguno"
COND = cond_A(NUDOS) if len(NUDOS) == N_BASES else np.inf
if NUDOS_FORZADOS is None and (OPTIMIZAR_NUDOS or COND >= UMBRAL_COND or len(NUDOS) != N_BASES):
    assert not ENSAYOS, f"con ensayos hacen falta los nudos con que se entrenó la red: copiá su archivo {ARCHIVO_BASES} a esta carpeta"
    print(f"⚙️  Buscando {N_BASES} nudos (origen: {_origen}; cond(A) = {COND:.1f}, umbral {UMBRAL_COND:g}). "
          "Evolución diferencial + Nelder–Mead: ≈ 25 min...")
    NUDOS = optimizar_nudos(N_BASES)
    COND = cond_A(NUDOS)
    _nuevos = True
    _origen = f"búsqueda nueva → se guardan en '{ARCHIVO_BASES}'"
assert len(NUDOS) == N_BASES, f"hacen falta {N_BASES} nudos"
print(f"Nudos ({_origen}): {NUDOS.tolist()}")
# Ensayos: la forma medida se escala para que en X_MAX valga lo mismo que la recta ideal (así una «lámina» conserva su tamaño);
#   lo que cambia es la FORMA de la curva entre 1,02 y 1,16.
_desvio = {}
for jm in ENSAYOS:
    j = jm - 1
    _f = leer_ensayo(ARCHIVO_ENSAYOS, jm)
    _le = pd.read_csv(ARCHIVO_ENSAYOS)[["lambda", f"material_{jm}"]].dropna()["lambda"]
    if _le.min() > X_MIN or _le.max() < X_MAX:
        print(f"⚠️  El ensayo del material {jm} ({ARCHIVO_ENSAYOS}) cubre λ de {_le.min():g} a {_le.max():g}: el rango de la red es "
              f"{X_MIN:g}–{X_MAX:g}. Fuera de lo medido la base se completa con 0 (abajo) o con una recta (arriba): conviene medir todo el rango.")
    _esc = (X_MAX - NUDOS[j]) / max(float(_f(np.array([X_MAX]))[0]), 1e-12)
    FORMA_ENSAYO[j] = (lambda f, e: (lambda l: f(l) * e))(_f, _esc)
    _r, _i = FORMA_ENSAYO[j](X), np.maximum(0.0, X - NUDOS[j])
    _desvio[j + 1] = 100 * np.sqrt(np.mean((_r - _i) ** 2)) / max(np.ptp(_i), 1e-12)
if ENSAYOS:
    COND = cond_A(NUDOS)
    print("Bases con ensayo (forma medida vs. recta ideal, desvío RMS en % del rango de la recta, entre X_MIN y X_MAX): "
          + ", ".join(f"material {j}: {e:.1f} %" for j, e in _desvio.items()))
assert COND < UMBRAL_COND, f"cond(A) = {COND:.1f} ≥ {UMBRAL_COND:g}: subí UMBRAL_COND o bajá N_BASES"
G = lineas(NUDOS)                                      # bases sin escalar (rectas ideales o curvas de ensayo)
_P = np.linalg.pinv(G)
PEND_MAX = np.round(np.maximum.reduce([(_P @ W_ESP.T).max(axis=1), (_P @ POB_Y[dentro_espacio(POB_Y)].T).max(axis=1), _P @ Y2]), 1)
_lw, _lp = C_LAM_RED * (_P @ W_ESP.T) / PEND_MAX[:, None], C_LAM_RED * (_P @ POB_Y.T) / PEND_MAX[:, None]
_neg, _negp = (_lw < -TOL_NEG).any(axis=0), (_lp < -TOL_NEG).any(axis=0)   # tolerancia: 0,01 lámina (de 10)
print(f"Signo de las láminas (MCO libre): curvas del espacio de trabajo con alguna lámina < 0: {_neg.sum()}/{len(W_ESP)} "
      f"(mínima {_lw.min():.3f}) | sujetos con alguna < 0: {_negp.sum()}/{len(POB_Y)} (mínima {_lp.min():.3f})")
if SIGNO_ESTRICTO:
    assert _neg.sum() == 0 and _negp.sum() == 0, "hay láminas negativas: poné OPTIMIZAR_NUDOS = True (o subí PESO_NEG) y volvé a correr"
assert np.size(C_LAM) in (1, N_BASES), f"C_LAM debe ser un número o una lista de {N_BASES} valores (uno por base)"
C_LAM = np.resize(np.asarray(C_LAM, float), N_BASES)
S = PEND_MAX / C_LAM                                   # pendiente de UNA lámina [kPa]
A = G * S                                              # bases del diseño
ESC = C_LAM / C_LAM_RED                                # láminas del diseño = láminas de la red × ESC
A_RED = G * (PEND_MAX / C_LAM_RED)                     # bases con que trabaja la red (C_LAM_RED láminas)


def mco(Y, B=A):
    """Mínimos cuadrados ordinarios, c = B⁺y (una curva o una por fila)."""
    return np.linalg.lstsq(B, np.asarray(Y, float).T, rcond=None)[0].T


def err_pct(Y, B):
    """(error punto a punto máximo, NRMSE) en % del rango, contra las curvas B."""
    Y, R = np.atleast_2d(Y), np.atleast_2d(Y) - np.atleast_2d(B)
    rg = np.ptp(Y, axis=1)
    return np.abs(R).max(axis=1) / rg * 100, np.sqrt(np.mean(R ** 2, axis=1)) / rg * 100


C_Y1, C_Y2 = mco(Y1), mco(Y2)
C_GEN_MAX = np.maximum(C_LAM_RED, (np.linalg.pinv(A_RED) @ POB_Y[MASK_TRAIN].T).max(axis=1))   # familias pura y combinada (red)

eM, eN = err_pct([Y1, Y2], mco([Y1, Y2]) @ A.T)
print(pd.DataFrame({"nudo k": NUDOS, "C_LAM": C_LAM, "S [kPa]": S, "láminas y1": C_Y1,
                    "láminas y2": C_Y2}, index=pd.Index(range(1, len(NUDOS) + 1), name="material")).round(4).to_string())
print(f"{N_BASES} bases | PEND_MAX = {PEND_MAX.tolist()} kPa | cond(A) = {COND:.2f} (umbral {UMBRAL_COND:g}) | MCO: y1 error máx. {eM[0]:.2f}% (NRMSE {eN[0]:.3f}%), "
      f"y2 {eM[1]:.2f}% ({eN[1]:.3f}%) | láminas de las familias pura y combinada (red): [0.1, {np.round(C_GEN_MAX, 2)}]")
pd.DataFrame(np.column_stack([X, A]), columns=["lambda"] + [f"A_{j + 1}" for j in range(N_BASES)]).to_csv(
    f"funciones_base_{VERSION}.csv", index=False)             # las curvas de las bases (una lámina de cada material)
if _nuevos:                                                  # nudos recién buscados: se guardan con la pendiente de una lámina (bases ideales)
    pd.DataFrame({"material": range(1, N_BASES + 1), "nudo": NUDOS, "pendiente_lamina_kPa": np.round(S, 4)}).to_csv(ARCHIVO_BASES, index=False)
    print(f"💾 Nudos y pendientes guardados en '{ARCHIVO_BASES}'")
if GUARDAR_PLANTILLA_ENSAYO:                                 # las bases ideales en el formato del archivo de ensayos
    _l = np.round(np.linspace(0.80, 1.25, 91), 4)
    pd.DataFrame({"lambda": _l, **{f"material_{j + 1}": np.round(S[j] * np.maximum(0.0, _l - NUDOS[j]) / 1e3, 6)
                                   for j in range(N_BASES)}}).to_csv(f"plantilla_ensayos_v40_{N_BASES}.csv", index=False)
    print(f"Plantilla: plantilla_ensayos_v40_{N_BASES}.csv (columnas lambda, material_1 … material_{N_BASES} [MPa]). Para usar ensayos "
          f"reales: guardar las columnas medidas en {ARCHIVO_ENSAYOS} (las que falten siguen ideales).")

## Celda 4 — Verificación de las bases sobre los 41 sujetos (MCO)

In [ ]:
# CELDA 4: VERIFICACIÓN DE LAS BASES SOBRE LOS 41 SUJETOS (piso de error del MCO, criterio y fabricabilidad)
import numpy as np
import pandas as pd
from scipy.optimize import nnls

C_POB = mco(POB_Y)
eM, eN = err_pct(POB_Y, C_POB @ A.T)
DF_PISO = POB[["id", "grupo", "test"]].assign(en_espacio=dentro_espacio(POB_Y), MaxAE_rel_pct=eM, NRMSE_pct=eN,
                                               fabricable=(C_POB >= -1e-9).all(axis=1), laminas_totales=C_POB.sum(axis=1),
                                               **{f"c{j+1}": C_POB[:, j] for j in range(len(NUDOS))})
grupos = {"Todos": DF_PISO.id.notna(), "Entrenamiento": ~DF_PISO.test, "Test": DF_PISO.test, "Dentro de y1–y2": DF_PISO.en_espacio,
          "Fuera de y1–y2": ~DF_PISO.en_espacio, "NT": DF_PISO.grupo == "NT", "HT": DF_PISO.grupo == "HT"}
print(pd.DataFrame({k: {"n": m.sum(), "NRMSE medio %": DF_PISO.NRMSE_pct[m].mean(), "NRMSE máx. %": DF_PISO.NRMSE_pct[m].max(),
                        "error pto. máx. %": DF_PISO.MaxAE_rel_pct[m].max(),
                        f"cumplen ≤{ERROR_MAX_PCT:g}%": (DF_PISO.MaxAE_rel_pct[m] <= ERROR_MAX_PCT).sum(),
                        "fabricables": DF_PISO.fabricable[m].sum()} for k, m in grupos.items()}).T.round(3).to_string())
_p = DF_PISO.loc[DF_PISO.MaxAE_rel_pct.idxmax()]
print(f"\nPeor sujeto: {_p.id}, {_p.MaxAE_rel_pct:.2f}% | láminas por material: mín. {C_POB.min(0).round(2)} | "
      f"máx. {C_POB.max(0).round(2)} (C_LAM = {C_LAM}) | totales: mediana {np.median(C_POB.sum(1)):.1f}")
for i in np.where(~DF_PISO.fabricable)[0]:
    j = int(np.argmin(C_POB[i]))
    print(f"   No fabricable {POB.id[i]}: c{j+1} = {C_POB[i, j]:+.3f} | imponer c ≥ 0 subiría el NRMSE de {eN[i]:.3f}% a "
          f"{err_pct(POB_Y[i], A @ nnls(A, POB_Y[i])[0])[1][0]:.3f}% | {'dentro' if _dent[i] else 'fuera'} de y1–y2")
PISO_MAXAE = float(eM.max())
DF_PISO.to_csv(f"verificacion_bases_sujetos_{VERSION}.csv", index=False)

## Celda 5 — Láminas de las curvas límite y error de los 41 sujetos (figura)

In [ ]:
# CELDA 5: y2 ARMADA CON LÁMINAS, LÁMINAS DE LAS CURVAS LÍMITE Y ERROR DE LOS 41 SUJETOS
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

col = plt.cm.viridis(np.linspace(0.0, 0.85, len(NUDOS)))
fig, ax = plt.subplots(1, 3, figsize=(19, 5.6), gridspec_kw=dict(width_ratios=[1.25, 1, 1.1]))
banda(ax[0])
acum = np.cumsum(np.column_stack([np.zeros_like(X), A * C_Y2]), axis=1)
for j in range(len(NUDOS)):
    ax[0].fill_between(X, acum[:, j], acum[:, j + 1], color=col[j], alpha=0.55, lw=0,
                       label=f"material {j+1} (k = {NUDOS[j]:.4f}): {C_Y2[j]:.2f} láminas")
for y, c, cl, n in [(Y2, C_Y2, "#b91c1c", "y2 (sintetizada)"), (Y1, C_Y1, "k", "y1 (sintetizada)")]:
    ax[0].plot(X, y, color=cl, lw=2.2, label=n)
    ax[0].plot(X, A @ c, "--", color="#f59e0b", lw=1.4)
ax[0].set(xlabel=ETQ_X, ylabel=ETQ_Y, title="y2 armada con láminas de cada material (- - ajuste MCO)")
ax[0].legend(fontsize=7.5, loc="upper left")

i = np.arange(1, len(NUDOS) + 1)
for d, c, cl, n in [(-0.19, C_Y1, "#6b7280", "y1"), (0.19, C_Y2, "#b91c1c", "y2")]:
    ax[1].bar(i + d, c, 0.38, color=cl, label=n)
    for ii, v in zip(i, c):
        ax[1].text(ii + d, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax[1].scatter(i, C_LAM, marker="_", s=900, color="k", label="C_LAM de cada material")
ax[1].set(xticks=i, ylabel="Láminas (MCO)", ylim=(0, 1.18 * max(C_LAM.max(), C_Y2.max())), title="Láminas de las curvas límite")
ax[1].legend(fontsize=8, loc="upper left")

d = DF_PISO.sort_values("MaxAE_rel_pct")
ax[2].barh(range(len(d)), d.MaxAE_rel_pct, color=np.where(d.en_espacio, "#2563eb", "#f59e0b"))
ax[2].set_yticks(range(len(d)), [f"{n}{' (test)' if t else ''}" for n, t in zip(d.id, d.test)], fontsize=6.5)
ax[2].axvline(ERROR_MAX_PCT, color="k", ls=":")
ax[2].set(xlabel="Error punto a punto máximo del MCO [% del rango]", title="Los 41 sujetos de Masson")
ax[2].legend(handles=[Patch(color="#2563eb", label="dentro de y1–y2"), Patch(color="#f59e0b", label="fuera de y1–y2")], fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"laminas_curvas_limite_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 6 — Librerías, entrada logarítmica, métricas y predicción

In [ ]:
# CELDA 6: LIBRERÍAS, TENSORFLOW, ENTRADA LOGARÍTMICA, MÉTRICAS, CONTROL DE CLASE Y PREDICCIÓN DE LA RED
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import tensorflow as tf
from scipy.optimize import nnls

np.random.seed(SEED)
tf.random.set_seed(SEED)
OFFSET_LOG = float(max(0.0, -min(Y1.min(), POB_Y.min())) + round(0.18 * np.median(POB_Y[:, -1])))   # entrada: log(y + OFFSET_LOG); la
#   constante crece con la escala de las curvas (σ ≈ 5 veces P): con 5 kPa el logaritmo aplastaba el tramo final, donde actúan los últimos materiales


def entrada_log(Y):
    return np.log(np.maximum(np.asarray(Y) + OFFSET_LOG, 1e-6))


def metricas(Y, B):
    """R², RMSE, MaxAE [kPa], NRMSE y MaxAE_rel [% del rango de cada curva]; una curva o una por fila."""
    Y, E = np.atleast_2d(Y), np.atleast_2d(Y) - np.atleast_2d(B)
    rg, rmse, maxae = np.ptp(Y, axis=1), np.sqrt(np.mean(E ** 2, axis=1)), np.abs(E).max(axis=1)
    m = dict(R2=1 - np.sum(E ** 2, axis=1) / np.sum((Y - Y.mean(axis=1, keepdims=True)) ** 2, axis=1),
             RMSE=rmse, MaxAE=maxae, NRMSE=rmse / rg * 100, MaxAE_rel=maxae / rg * 100)
    return {k: (float(v[0]) if len(v) == 1 else v) for k, v in m.items()}


def predecir(Y):
    """Láminas de cada material (diseño, C_LAM) que predice la red para una curva o una por fila."""
    c = sy.inverse_transform(model.predict(sx.transform(entrada_log(np.atleast_2d(Y))), batch_size=1024, verbose=0)) * ESC
    return c[0] if np.ndim(Y) == 1 else c


def panel(ax, s, titulo):
    """Curva de test #s: entrada, óptimo MCO y red (la usan las Celdas 13, 16 y 18)."""
    banda(ax)
    ax.plot(X, B_IN[s], "k-", lw=2, label="Curva de entrada")
    ax.plot(X, B_OPT[s], "b:", lw=1.8, label="Óptimo MCO")
    ax.plot(X, B_RN[s], "r--", lw=1.8, label="Red")
    ax.set_ylim(-0.03 * max(B_IN[s].max(), 1), 1.15 * max(B_IN[s].max(), 1))
    ax.set_title(titulo, fontsize=8.5, fontweight="bold")
    ax.set(xlabel=ETQ_X, ylabel=ETQ_Y)
    ax.grid(alpha=0.4, ls=":")
    ax.legend(fontsize=7, loc="upper left")


_K = np.linspace(NUDO_MIN_CLASE, X_MAX, N_NUDOS_CLASE + 2)[:-1]
_B_CLASE = np.maximum(0.0, X[:, None] - _K)[::3] / (X_MAX - _K)


def distancia_clase(y):
    """Distancia [% del rango] de y a las curvas realizables con capas (combinaciones c ≥ 0 de 40 líneas truncadas)."""
    ys = np.asarray(y)[::3]
    return float(np.sqrt(np.mean((ys - _B_CLASE @ nnls(_B_CLASE, ys)[0]) ** 2)) / np.ptp(y) * 100)


print(f"✅ TensorFlow {tf.__version__} ({'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'}) | OFFSET_LOG = {OFFSET_LOG:g} | "
      f"control de clase: distancia ≤ {TOL_CLASE_NRMSE}%")

## Celda 7 — Dataset sintético y etiquetas enteras (minimax MILP sobre cada curva)

In [ ]:
# CELDA 7: DATASET SINTÉTICO — V40: etiqueta = láminas ENTERAS del redondeo minimax (MILP, HiGHS) de la curva que ve la red
#   (en la escala del diseño, C_LAM = 20, y pasadas a láminas de la red: z / ESC). Antes (V39): láminas reales del MCO.
#   0 pura: un material, láminas log-uniformes en [0,1; C_GEN_MAX_j]   1 combinada: 2–N materiales, U(0, C_GEN_MAX_j)
#   + las familias fisiológicas de la Celda 2
import numpy as np
import pandas as pd

C_MIN_PURA, RUIDO_PURAS = 0.10, 0.005
rng_b = np.random.default_rng(2028)
nb = len(NUDOS)
c0 = np.zeros((N_FAM[0], nb))
j0 = rng_b.permutation(np.arange(N_FAM[0]) % nb)
c0[np.arange(N_FAM[0]), j0] = np.exp(rng_b.uniform(np.log(C_MIN_PURA), np.log(C_GEN_MAX[j0])))
c1 = np.zeros((N_FAM[1], nb))
for i in range(N_FAM[1]):
    act = rng_b.choice(nb, size=rng_b.integers(2, nb + 1), replace=False)
    c1[i, act] = rng_b.uniform(0.0, 1.0, size=len(act)) * C_GEN_MAX[act]
c1 *= np.minimum(1.0, Y_TECHO / np.maximum((c1 @ A_RED.T).max(axis=1, keepdims=True), 1e-9))
for f, c in ((0, c0), (1, c1)):
    LIMPIAS[f] = c @ A_RED.T
    RUIDOSAS[f] = ruido(LIMPIAS[f], RUIDO_PURAS, rng_b)

Y_limpio, Y_raw = (np.vstack([D[f] for f in sorted(FAMILIAS)]) for D in (LIMPIAS, RUIDOSAS))
familia = np.concatenate([np.full(N_FAM[f], f, np.int8) for f in sorted(FAMILIAS)])
C_RED_MCO = mco(Y_raw, A_RED)                                       # láminas reales del MCO (solo para comparar)
import os, sys, time, contextlib
from scipy.optimize import milp, LinearConstraint, Bounds


@contextlib.contextmanager
def _sin_texto():
    sys.stdout.flush()
    fd, nul = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    os.dup2(nul, 1)
    try:
        yield
    finally:
        os.dup2(fd, 1); os.close(fd); os.close(nul)


def minimax_entero(y, B, z0=None):
    """Redondeo minimax por MILP (HiGHS), 500 puntos de λ, sin acotar la búsqueda (óptimo global)."""
    Bs, ys = B[::2], y[::2]
    m, n = Bs.shape
    lo, hi = np.zeros(n + 1), np.full(n + 1, np.inf)
    with _sin_texto():
        r = milp(np.r_[np.zeros(n), 1.0], integrality=np.r_[np.ones(n), 0], bounds=Bounds(lo, hi),
                 constraints=LinearConstraint(np.vstack([np.c_[Bs, -np.ones(m)], np.c_[-Bs, -np.ones(m)]]), -np.inf, np.r_[ys, -ys]),
                 options=dict(time_limit=30))
    if r.x is None:                                                 # HiGHS no llegó a una solución en 30 s:
        z0 = np.zeros(n) if z0 is None else np.maximum(np.round(z0), 0)  # se reintenta en una caja ±6 alrededor del MCO
        lo[:n], hi[:n] = np.maximum(z0 - 6, 0), z0 + 6
        with _sin_texto():
            r = milp(np.r_[np.zeros(n), 1.0], integrality=np.r_[np.ones(n), 0], bounds=Bounds(lo, hi),
                     constraints=LinearConstraint(np.vstack([np.c_[Bs, -np.ones(m)], np.c_[-Bs, -np.ones(m)]]), -np.inf, np.r_[ys, -ys]),
                     options=dict(time_limit=120))
        FALLOS_MILP.append(1)
        if r.x is None:
            return z0
    return np.round(r.x[:n])


FALLOS_MILP = []


if os.path.exists(ARCHIVO_ETIQUETAS_ENTERAS):
    Z_ENT = np.load(ARCHIVO_ETIQUETAS_ENTERAS)["Z"]
    print(f"📦 Etiquetas enteras cargadas de '{ARCHIVO_ETIQUETAS_ENTERAS}'")
else:
    _parcial = ARCHIVO_ETIQUETAS_ENTERAS.replace(".npz", "_parcial.npz")
    Z_ENT, _i0 = np.zeros_like(C_RED_MCO), 0
    if os.path.exists(_parcial):                                    # si se cortó, sigue desde donde quedó
        _p = np.load(_parcial)
        _i0 = int(_p["n"]); Z_ENT[:_i0] = _p["Z"][:_i0]
    _t0, _C20 = time.time(), C_RED_MCO * ESC
    print(f"⏳ Redondeo minimax de {N_SAMPLES} curvas (se guarda cada 2000; ≈ 60 min)...", flush=True)
    for i in range(_i0, N_SAMPLES):
        Z_ENT[i] = minimax_entero(Y_raw[i], A, _C20[i])
        if (i + 1) % 2000 == 0:
            np.savez(_parcial, Z=Z_ENT, n=i + 1)
            print(f"   {i + 1}/{N_SAMPLES} | {(time.time() - _t0) / 60:.1f} min", flush=True)
    np.savez_compressed(ARCHIVO_ETIQUETAS_ENTERAS, Z=Z_ENT, nudos=NUDOS)
    if os.path.exists(_parcial):
        os.remove(_parcial)
    print(f"   Curvas que necesitaron reintento en caja ±6: {len(FALLOS_MILP)}")
C_RED = Z_ENT / ESC                                                 # etiquetas: láminas enteras (escala C_LAM) en láminas de la red
_e_ent = err_pct(Y_raw, Z_ENT @ A.T)[0]
print(f"Etiquetas enteras: error del redondeo minimax contra la curva que ve la red, mediana {np.median(_e_ent):.2f} %, "
      f"p95 {np.percentile(_e_ent, 95):.2f} % | |entero − MCO| medio {np.abs(Z_ENT - C_RED_MCO * ESC).mean():.2f} láminas")
MASCARA = np.zeros_like(C_RED, dtype=np.float32)                    # crosstalk: materiales que el generador dejó en 0
MASCARA[:N_FAM[0] + N_FAM[1]] = np.vstack([c0, c1]) == 0
X_log = entrada_log(Y_raw)

_C0 = mco(Y_limpio, A)
_eM = err_pct(Y_limpio, _C0 @ A.T)[0]
print(f"✅ Dataset: {N_SAMPLES} curvas | etiquetas = láminas ENTERAS (minimax, MILP, HiGHS)")
print(pd.DataFrame({FAMILIAS[f]: {"curvas": N_FAM[f], "puntos fuera de y1–y2 %": 100 * np.mean((Y_raw[m] < Y1) | (Y_raw[m] > Y2)),
                                  "aceptación %": 100 * ACEPTACION.get(f, 1.0), f"MCO cumple ≤{ERROR_MAX_PCT:g}% %": 100 * np.mean(_eM[m] <= ERROR_MAX_PCT),
                                  "error MCO máx. %": _eM[m].max(), "fabricables %": 100 * np.mean((C_RED[m] >= -1e-9).all(1))}
                    for f in FAMILIAS for m in [familia == f]}).T.round(2).to_string())
_esp = np.isin(familia, (2, 3, 5))
print(f"Espacio de trabajo ({_esp.sum()} curvas de Masson dentro de y1–y2): máximo de láminas por material "
      f"{_C0[_esp].max(0).round(2)} (C_LAM = {C_LAM})")

## Celda 8 — Partición y normalización

In [ ]:
# CELDA 8: PARTICIÓN TRAIN / VAL / TEST (estratificada por familia) Y NORMALIZACIÓN
# Con ENTRENAR = False se usan los escaladores guardados con la red; deben coincidir con los de este dataset.
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

idx_trval, idx_test = train_test_split(np.arange(N_SAMPLES), test_size=0.20, random_state=SEED, stratify=familia)
idx_train, idx_val = train_test_split(idx_trval, test_size=0.125, random_state=SEED, stratify=familia[idx_trval])
sx, sy = StandardScaler().fit(X_log[idx_train]), StandardScaler().fit(C_RED[idx_train])
if not ENTRENAR:
    with open(ARCHIVO_SCALERS, "rb") as f:
        meta = pickle.load(f)
    dif = max(np.abs(meta["scaler_x"].mean_ - sx.mean_).max(), np.abs(meta["scaler_y"].mean_ - sy.mean_).max())
    sx, sy = meta["scaler_x"], meta["scaler_y"]
    assert meta["offset_log"] == OFFSET_LOG, "OFFSET_LOG distinto del de la red guardada"
    assert np.allclose(meta["nudos"], NUDOS), f"la red se entrenó con los nudos {np.round(meta['nudos'], 4).tolist()}"
    assert "bases" not in meta or np.allclose(meta["bases"], A_RED, rtol=1e-6, atol=1e-9), \
        "la red se entrenó con otras bases (otros ensayos): borrá sus archivos o poné ENTRENAR = True"
    print(f"📦 Escaladores de '{ARCHIVO_SCALERS}' | diferencia con este dataset: {dif:.1e} (≈ 0 ⇒ mismo dataset de entrenamiento)")
X_train, X_val, X_test = (sx.transform(X_log[i]) for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = (sy.transform(C_RED[i]) for i in (idx_train, idx_val, idx_test))
m_train, m_val = MASCARA[idx_train], MASCARA[idx_val]
print(f"✅ Train {len(idx_train)} | Val {len(idx_val)} | Test {len(idx_test)} | sy.mean_ {sy.mean_.round(4)} | "
      f"sy.scale_ {sy.scale_.round(4)} | crosstalk penalizado en {MASCARA.mean():.1%} de los coeficientes")

## Celda 9 — Pérdida y arquitectura

In [ ]:
# CELDA 9: PÉRDIDA Y ARQUITECTURA
# L = 1·MSE(láminas estandarizadas) + 5·MSE(A·ĉ − A·c, relativo al máximo de cada curva) + PESO_CROSSTALK·media(ĉ² en materiales ausentes)
import numpy as np
import tensorflow as tf

PESO_COEF, PESO_REC, PESO_CROSSTALK = 1.0, 5.0, 5.0      # 5 con C_LAM_RED = 10 (peso efectivo ≈ 14, la mejor red: V22)
CAPAS_OCULTAS, L2 = (256, 128, 64), 1e-5
nb = len(NUDOS)
_A_tf, _mu, _sd = (tf.constant(v, tf.float32) for v in (A_RED.T, sy.mean_, sy.scale_))


def perdida(y_aug, y_pred):
    y_true, mascara = y_aug[:, :nb], y_aug[:, nb:]                  # etiqueta | máscara de crosstalk
    c_true, c_pred = y_true * _sd + _mu, y_pred * _sd + _mu
    b_true, b_pred = c_true @ _A_tf, c_pred @ _A_tf
    y_max = tf.maximum(tf.reduce_max(b_true, axis=1, keepdims=True), 1.0)
    return (PESO_COEF * tf.reduce_mean(tf.square(y_true - y_pred))
            + PESO_REC * tf.reduce_mean(tf.square((b_true - b_pred) / (y_max + 1e-6)))
            + PESO_CROSSTALK * tf.reduce_mean(tf.square(c_pred) * mascara))


def mae(y_aug, y_pred):
    return tf.reduce_mean(tf.abs(y_aug[:, :nb] - y_pred))


capas = [tf.keras.Input(shape=(N_PTS,))]
for u in CAPAS_OCULTAS:
    capas += [tf.keras.layers.Dense(u, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(L2)),
              tf.keras.layers.LayerNormalization()]
model = tf.keras.Sequential(capas + [tf.keras.layers.Dense(nb)])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss=perdida, metrics=[mae])
print(f"✅ Red {N_PTS} → {' → '.join(map(str, CAPAS_OCULTAS))} → {nb} (ReLU + LayerNormalization) | "
      f"{model.count_params():,} parámetros | pesos de la pérdida: {PESO_COEF:g} / {PESO_REC:g} / {PESO_CROSSTALK:g}")

## Celda 10 — Integridad de los datos

In [ ]:
# CELDA 10: INTEGRIDAD DE LOS DATOS
import numpy as np

for k, v in dict(X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test).items():
    assert np.isfinite(v).all(), f"{k} tiene NaN o Inf"
print(f"✅ Sin NaN ni Inf | etiquetas en [{C_RED.min():.3f}, {C_RED.max():.3f}] láminas | "
      f"curvas con algún c < 0: {np.mean(C_RED.min(1) < 0):.1%}")

## Celda 11 — Entrenamiento o carga de la red

In [ ]:
# CELDA 11: ENTRENAMIENTO (ENTRENAR = True) O CARGA DE LA RED ENTRENADA (False)
#   CONTINUAR = True (con la red cargada): sigue entrenando desde sus pesos con la tasa de aprendizaje reiniciada en LR_REINICIO;
#   solo guarda si baja la pérdida de validación. Poner de nuevo en False después de usarlo.
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks as cb
from tensorflow.keras.models import load_model

EPOCHS, BATCH_SIZE, MIN_DELTA = 3000, 256, 1e-5
CONTINUAR, EPOCAS_EXTRA, LR_REINICIO = False, 2000, 1e-4


def _callbacks():
    return [cb.TerminateOnNaN(), cb.EarlyStopping(patience=150, min_delta=MIN_DELTA, restore_best_weights=True),
            cb.ReduceLROnPlateau(factor=0.5, patience=50, min_delta=MIN_DELTA, min_lr=1e-6)]


if ENTRENAR:                                     # si el entrenamiento se corta, al volver a correr la celda sigue desde la última época
    import os, shutil
    t0, _resp, _log = time.time(), f"_respaldo_{VERSION}", f"_registro_{VERSION}.csv"
    model.fit(X_train, np.hstack([y_train, m_train]), validation_data=(X_val, np.hstack([y_val, m_val])), epochs=EPOCHS,
              batch_size=BATCH_SIZE, verbose=2, callbacks=_callbacks() + [cb.BackupAndRestore(_resp), cb.CSVLogger(_log, append=True)])
    hist = pd.read_csv(_log).drop_duplicates("epoch", keep="last").drop(columns="epoch")
    hist.index = pd.RangeIndex(1, len(hist) + 1, name="epoca")
    shutil.rmtree(_resp, ignore_errors=True)
    os.remove(_log)
    model.save(ARCHIVO_MODELO)
    with open(ARCHIVO_SCALERS, "wb") as f:
        pickle.dump(dict(scaler_x=sx, scaler_y=sy, offset_log=OFFSET_LOG, c_lam_red=C_LAM_RED, nudos=NUDOS, bases=A_RED,
                         ensayos=list(ENSAYOS), version=VERSION), f)
    hist.to_csv(ARCHIVO_HISTORIAL)
    print(f"✅ Entrenada en {(time.time() - t0) / 60:.1f} min ({len(hist)} épocas) → {ARCHIVO_MODELO}, {ARCHIVO_SCALERS}")
else:
    model = load_model(ARCHIVO_MODELO, compile=False)
    hist = pd.read_csv(ARCHIVO_HISTORIAL, index_col="epoca")
    print(f"📦 Red cargada de '{ARCHIVO_MODELO}' (sin reentrenar; {len(hist)} épocas de entrenamiento)")
    if CONTINUAR:
        import tensorflow as tf
        Yv = np.hstack([y_val, m_val])
        model.compile(optimizer=tf.keras.optimizers.Adam(LR_REINICIO, clipnorm=1.0), loss=perdida, metrics=[mae])
        v0, n0, t0 = model.evaluate(X_val, Yv, batch_size=1024, verbose=0)[0], len(hist), time.time()
        h = model.fit(X_train, np.hstack([y_train, m_train]), validation_data=(X_val, Yv), initial_epoch=n0,
                      epochs=n0 + EPOCAS_EXTRA, batch_size=BATCH_SIZE, verbose=2, callbacks=_callbacks())
        v1 = model.evaluate(X_val, Yv, batch_size=1024, verbose=0)[0]
        print(f"   Continuación: val_loss {v0:.6f} → {v1:.6f} ({(v1 / v0 - 1) * 100:+.1f} %), {len(h.history['loss'])} épocas en "
              f"{(time.time() - t0) / 60:.1f} min")
        if v1 < v0:
            hist = pd.concat([hist, pd.DataFrame(h.history, index=pd.RangeIndex(n0 + 1, n0 + len(h.history["loss"]) + 1, name="epoca"))])
            model.save(ARCHIVO_MODELO)
            hist.to_csv(ARCHIVO_HISTORIAL)
            print(f"   ✅ Mejoró: guardada en {ARCHIVO_MODELO} ({len(hist)} épocas en total)")
        else:
            model = load_model(ARCHIVO_MODELO, compile=False)
            print("   ↩️ No mejoró: se mantiene la red anterior")

v = hist["val_loss"].to_numpy()
mejor, ep = np.inf, len(v)                       # época que restaura EarlyStopping (última mejora > MIN_DELTA)
for i, x_ in enumerate(v):
    if x_ + MIN_DELTA < mejor:
        mejor, ep = x_, i + 1
ep_conv = int(np.argmax(v <= v.min() * 1.01)) + 1
print(f"   Pesos de la época {ep}: val_loss {v[ep - 1]:.6f} | loss {hist['loss'].iloc[ep - 1]:.6f} | "
      f"convergencia práctica (≤ 1 % del mínimo) en la época {ep_conv}")
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(hist.index, hist["loss"], label="Entrenamiento")
ax.semilogy(hist.index, v, "--", label="Validación")
ax.axvline(ep, color="gray", ls=":", label=f"Pesos guardados ({ep})")
lr = next((c for c in ("learning_rate", "lr") if c in hist), None)
if lr:
    ax.twinx().semilogy(hist.index, hist[lr], color="#6b7280", lw=1, alpha=0.6)
ax.set(xlabel="Época", ylabel="Pérdida", title=f"Evolución de la pérdida ({ARCHIVO_MODELO}; gris: learning rate)")
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(f"evolucion_perdida_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 12 — Evaluación base en test

In [ ]:
# CELDA 12: EVALUACIÓN BASE EN TEST (la usan las Celdas 13, 16 y 18)
# Todo en láminas del diseño (C_LAM). La red se evalúa sin pulido; el pulido en MCO es c = A⁺y (ecuaciones normales).
import time
import numpy as np

t0 = time.perf_counter()
C_RN = predecir(Y_raw[idx_test])
t_rn = (time.perf_counter() - t0) / len(idx_test)
C_OPT = C_RED[idx_test] * ESC                                     # etiqueta = MCO de la curva de entrada
FAM_T, B_IN, B_LIMPIO = familia[idx_test], Y_raw[idx_test], Y_limpio[idx_test]
B_RN, B_OPT = C_RN @ A.T, C_OPT @ A.T
t0 = time.perf_counter()
for y in B_IN[:300]:
    np.linalg.lstsq(A, y, rcond=None)
t_mco = (time.perf_counter() - t0) / 300
g = np.abs((A.T @ (B_OPT - B_IN).T)).max(0) / np.abs(A.T @ B_IN.T).max(0)
print(f"✅ Test: {len(idx_test)} curvas | láminas negativas: red {np.mean(C_RN.min(1) < 0):.1%} de las curvas "
      f"(mín. {C_RN.min():.3f}), MCO {np.mean(C_OPT.min(1) < 0):.1%} (mín. {C_OPT.min():.3f})")
print(f"   Pulido (MCO): ecuaciones normales |Aᵀ(Ac − y)| / |Aᵀy| ≤ {g.max():.1e} ⇒ red + pulido = óptimo exacto")
print(f"   Tiempo por curva: red en lote {t_rn * 1e3:.4f} ms | MCO {t_mco * 1e3:.4f} ms")

## Celda 13 — Reconstrucción en test (una muestra por familia)

In [ ]:
# CELDA 13: RECONSTRUCCIÓN EN TEST (una muestra por familia) Y ERROR CONTRA LA CURVA SIN RUIDO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


rng_plot = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 3, figsize=(19.5, 13.8))
for ax, f in zip(axes.flat, sorted(FAMILIAS)):
    s = int(rng_plot.choice(np.where(FAM_T == f)[0]))
    mr, mo = metricas(B_LIMPIO[s], B_RN[s]), metricas(B_LIMPIO[s], B_OPT[s])
    panel(ax, s, f"Test #{s} — {FAMILIAS[f]} | vs. curva limpia: NRMSE red {mr['NRMSE']:.3f}% · óptimo {mo['NRMSE']:.3f}%\n"
                 f"red [{', '.join(f'{v:.2f}' for v in C_RN[s])}]\nMCO [{', '.join(f'{v:.2f}' for v in C_OPT[s])}]")
for ax in axes.flat[len(FAMILIAS):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"reconstruccion_curvas_test_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

M0_RN, M0_OPT, MI_RN = metricas(B_LIMPIO, B_RN), metricas(B_LIMPIO, B_OPT), metricas(B_IN, B_RN)
print("NRMSE medio [%] por familia:")
print(pd.DataFrame({FAMILIAS[f]: {"red vs. limpia": M0_RN["NRMSE"][m].mean(), "óptimo vs. limpia": M0_OPT["NRMSE"][m].mean(),
                                  "red vs. entrada con ruido": MI_RN["NRMSE"][m].mean()}
                    for f in FAMILIAS for m in [FAM_T == f]}).T.round(3).to_string())

## Celda 14 — Validación con los 41 sujetos y las 4 curvas medias

In [ ]:
# CELDA 14: VALIDACIÓN CON LOS 41 SUJETOS REALES Y LAS 4 CURVAS MEDIAS (red vs. óptimo MCO, sin ruido)
import numpy as np
import pandas as pd

C_SUJ_RN, C_SUJ_OPT = predecir(POB_Y), mco(POB_Y)
MS_RN, MS_OPT = metricas(POB_Y, C_SUJ_RN @ A.T), metricas(POB_Y, C_SUJ_OPT @ A.T)
DF_SUJ = POB[["id", "grupo", "test"]].assign(R2_red=MS_RN["R2"], R2_opt=MS_OPT["R2"], NRMSE_red=MS_RN["NRMSE"],
                                             NRMSE_opt=MS_OPT["NRMSE"], MaxAE_rel_red=MS_RN["MaxAE_rel"],
                                             dc_max=np.abs(C_SUJ_RN - C_SUJ_OPT).max(1),
                                             en_clase=[distancia_clase(y) <= TOL_CLASE_NRMSE for y in POB_Y])
DF_SUJ["exceso_pct"] = (DF_SUJ.NRMSE_red / DF_SUJ.NRMSE_opt - 1) * 100
DF_SUJ.to_csv(f"validacion_sujetos_reales_{VERSION}.csv", index=False)
print(pd.DataFrame({k: {"n": m.sum(), "R² red mín.": DF_SUJ.R2_red[m].min(), "NRMSE óptimo %": DF_SUJ.NRMSE_opt[m].mean(),
                        "NRMSE red %": DF_SUJ.NRMSE_red[m].mean(),
                        "exceso %": (DF_SUJ.NRMSE_red[m].mean() / DF_SUJ.NRMSE_opt[m].mean() - 1) * 100,
                        "error pto. red máx. %": DF_SUJ.MaxAE_rel_red[m].max(),
                        f"red cumple ≤{ERROR_MAX_PCT:g}%": (DF_SUJ.MaxAE_rel_red[m] <= ERROR_MAX_PCT).sum(), "máx |Δc|": DF_SUJ.dc_max[m].max()}
                    for k, m in {"Test": DF_SUJ.test, "Entrenamiento": ~DF_SUJ.test, "NT": DF_SUJ.grupo == "NT",
                                 "HT": DF_SUJ.grupo == "HT", "Todos": DF_SUJ.id.notna()}.items()}).T.round(4).to_string())
_t = DF_SUJ[DF_SUJ.test]
print(f"Peor R² de la red: {DF_SUJ.id[DF_SUJ.R2_red.idxmin()]} | exceso en test: mediana {_t.exceso_pct.median():+.1f}%, "
      f"máximo {_t.exceso_pct.max():+.1f}% ({_t.id[_t.exceso_pct.idxmax()]}) | control de clase: {DF_SUJ.en_clase.mean():.0%} aceptados")

RES_MEDIAS = {}
for k, y in MEDIAS.items():
    c_rn, c_opt = predecir(y), mco(y)
    RES_MEDIAS[k] = dict(y=y, b_rn=A @ c_rn, b_opt=A @ c_opt, m_rn=metricas(y, A @ c_rn), m_opt=metricas(y, A @ c_opt))
    print(f"\n{k} (n = {GRUPOS_MEDIAS[k].sum()}): láminas MCO {c_opt.round(4)} (total {c_opt.sum():.2f}) | red {c_rn.round(4)}")
    print(f"   NRMSE red {RES_MEDIAS[k]['m_rn']['NRMSE']:.4f}% / óptimo {RES_MEDIAS[k]['m_opt']['NRMSE']:.4f}% | "
          f"error pto. red {RES_MEDIAS[k]['m_rn']['MaxAE_rel']:.3f}% / óptimo {RES_MEDIAS[k]['m_opt']['MaxAE_rel']:.3f}% | "
          f"exceso de RMSE {(RES_MEDIAS[k]['m_rn']['RMSE'] / RES_MEDIAS[k]['m_opt']['RMSE'] - 1) * 100:+.2f}%")

## Celda 15 — Figuras: curvas medias y sujetos de test

In [ ]:
# CELDA 15: FIGURAS DE VALIDACIÓN — CURVAS MEDIAS (con residuos) Y SUJETOS DE TEST
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
for (k, r), a, b in zip(RES_MEDIAS.items(), axes[0], axes[1]):
    banda(a)
    a.plot(X, r["y"], "k-", lw=2.5, label="Media reconstruida")
    a.plot(X, r["b_rn"], "r--", lw=2, label="Red")
    a.plot(X, r["b_opt"], "b:", lw=1.6, label="Óptimo MCO")
    a.set(title=f"{k}\nNRMSE red {r['m_rn']['NRMSE']:.2f}% (óptimo {r['m_opt']['NRMSE']:.2f}%)", ylabel=ETQ_Y)
    a.legend(fontsize=7, loc="upper left")
    b.plot(X, r["y"] - r["b_rn"], "r-", lw=1.5, label="Residuo red")
    b.plot(X, r["y"] - r["b_opt"], "b--", lw=1.5, label="Residuo óptimo")
    for k_ in NUDOS[NUDOS >= X_MIN]:
        b.axvline(k_, color="gray", ls="-.", alpha=0.5)
    b.set(xlabel=ETQ_X, ylabel="Residuo [kPa]")
    b.legend(fontsize=7)
for a in axes.flat:
    a.grid(alpha=0.3)
plt.suptitle("Curvas medias de Masson et al. (2011) — líneas grises: nudos", fontweight="bold")
plt.tight_layout()
plt.savefig(f"validacion_curvas_medias_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(21, 6.8))
for ax, i in zip(axes.flat, np.where(MASK_TEST)[0]):
    banda(ax)
    ax.plot(X, POB_Y[i], "k-", lw=2, label="Sujeto")
    ax.plot(X, A @ C_SUJ_RN[i], "r--", lw=1.6, label="Red")
    ax.plot(X, A @ C_SUJ_OPT[i], "b:", lw=1.4, label="Óptimo")
    ax.set_title(f"{POB.id[i]} — $R^2$ red {MS_RN['R2'][i]:.4f} / ópt. {MS_OPT['R2'][i]:.4f}", fontsize=9, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="upper left")
plt.suptitle("Sujetos reservados para test (no usados en el generador ni en las bases)", fontweight="bold")
plt.tight_layout()
plt.savefig(f"validacion_sujetos_test_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 16 — Evaluación en test por familia, criterio y peores casos

In [ ]:
# CELDA 16: EVALUACIÓN EN TEST — LÁMINAS, EXCESO DE LA RED SOBRE EL ÓPTIMO, CRITERIO Y PEORES CASOS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

M_RN, M_OPT = metricas(B_IN, B_RN), metricas(B_IN, B_OPT)
E0_RN, E0_OPT = metricas(B_LIMPIO, B_RN)["MaxAE_rel"], metricas(B_LIMPIO, B_OPT)["MaxAE_rel"]
fisio = np.isin(FAM_T, FAMILIAS_OBJETIVO)
for titulo, m in [("todas las familias", np.ones_like(fisio)), ("familias fisiológicas", fisio)]:
    d = C_RN[m] - C_OPT[m]
    print(f"Láminas, {titulo} (n = {m.sum()}): R² {np.round(1 - (d ** 2).sum(0) / ((C_OPT[m] - C_OPT[m].mean(0)) ** 2).sum(0), 6)} | "
          f"MAE {np.abs(d).mean(0).round(4)}")

RESUMEN = pd.DataFrame({FAMILIAS[f]: {"n": m.sum(), "NRMSE óptimo %": M_OPT["NRMSE"][m].mean(), "NRMSE red %": M_RN["NRMSE"][m].mean(),
                                      "exceso %": (M_RN["NRMSE"][m].mean() / M_OPT["NRMSE"][m].mean() - 1) * 100,
                                      "R² red mín.": M_RN["R2"][m].min(), "peor (test #)": int(np.where(m)[0][np.argmin(M_RN["R2"][m])]),
                                      f"red cumple ≤{ERROR_MAX_PCT:g}% %": 100 * np.mean(E0_RN[m] <= ERROR_MAX_PCT),
                                      "red máx. %": E0_RN[m].max(), f"MCO cumple ≤{ERROR_MAX_PCT:g}% %": 100 * np.mean(E0_OPT[m] <= ERROR_MAX_PCT),
                                      "MCO máx. %": E0_OPT[m].max()} for f in FAMILIAS for m in [FAM_T == f]}).T
print("\nReconstrucción contra la curva de entrada; criterio contra la curva sin ruido. Exceso = NRMSE red / NRMSE óptimo − 1.")
print(RESUMEN.round(3).to_string())
RESUMEN.to_csv(f"resumen_test_por_familia_{VERSION}.csv")
pd.DataFrame(np.column_stack([idx_test, FAM_T, M_RN["NRMSE"], M_OPT["NRMSE"], E0_RN, C_OPT, C_RN]),
             columns=["idx", "familia", "NRMSE_red", "NRMSE_opt", "MaxAE_rel_red_limpia"]
             + [f"c{j+1}_{t}" for t in ("mco", "red") for j in range(len(NUDOS))]).to_csv(f"metricas_test_por_curva_{VERSION}.csv", index=False)

COL = dict(zip(sorted(FAMILIAS), ["#d62728", "#2ca02c", "#1f77b4", "#ff7f0e", "#9467bd", "#8c564b", "#17becf"]))
for nombre, fams in [("todas", sorted(FAMILIAS)), ("fisiologicas", FAMILIAS_OBJETIVO)]:
    fig, axes = plt.subplots(1, len(NUDOS), figsize=(4.4 * len(NUDOS), 4.2))
    for j, ax in enumerate(axes):
        for f in fams:
            m = FAM_T == f
            ax.scatter(C_OPT[m, j], C_RN[m, j], s=10, alpha=0.35, color=COL[f], edgecolors="none", label=FAMILIAS[f])
        lim = [C_OPT[np.isin(FAM_T, fams), j].min(), C_OPT[np.isin(FAM_T, fams), j].max()]
        ax.plot(lim, lim, "k--", lw=1.2)
        ax.set(title=f"c{j+1} — {nombre}", xlabel="MCO (etiqueta)", ylabel="Red")
        ax.grid(alpha=0.4, ls="--")
    axes[0].legend(fontsize=7, markerscale=2)
    plt.tight_layout()
    plt.savefig(f"diagnostico_coeficientes_{nombre}_{VERSION}.png", dpi=300, bbox_inches="tight")
    plt.show()

ymax = np.maximum(B_IN.max(1, keepdims=True), 1.0)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5))
le = np.log10(np.clip(1 - M_RN["R2"], 1e-8, None))
for f in FAMILIAS:
    a1.hist(le[FAM_T == f], bins=np.linspace(le.min(), le.max(), 50), alpha=0.55, color=COL[f], label=FAMILIAS[f])
a1.set(yscale="log", xlabel=r"$\log_{10}(1 - R^2)$ red vs. entrada (izquierda = mejor)", ylabel="Curvas")
for d, c, n in [((B_IN - B_RN) / ymax * 100, "#9467bd", "entrada − red (total)"), ((B_OPT - B_RN) / ymax * 100, "#ef4444", "óptimo − red (atribuible a la red)")]:
    a2.hist(d.ravel(), bins=200, density=True, alpha=0.5, color=c, label=n)
a2.set(yscale="log", xlabel="Residuo [% del máximo de cada curva]", ylabel="Densidad")
for a in (a1, a2):
    a.legend(fontsize=8)
    a.grid(alpha=0.4, ls="--")
plt.tight_layout()
plt.savefig(f"distribucion_residuos_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, len(FAMILIAS), figsize=(5.2 * len(FAMILIAS), 9))
for c, f in enumerate(FAMILIAS):
    m = np.where(FAM_T == f)[0]
    for r, (s, crit) in enumerate([(m[np.argmin(M_RN["R2"][m])], "R²"), (m[np.argmax(M_RN["MaxAE"][m])], "MaxAE")]):
        panel(axes[r, c], s, f"Peor por {crit} — {FAMILIAS[f]} (test #{s})\nred: R² {M_RN['R2'][s]:.4f}, MaxAE {M_RN['MaxAE'][s]:.2f} kPa"
                             f" | óptimo: {M_OPT['R2'][s]:.4f}, {M_OPT['MaxAE'][s]:.2f} kPa")
plt.tight_layout()
plt.savefig(f"peores_casos_test_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 17 — Ley de Fung (nunca vista) y control de clase

In [ ]:
# CELDA 17: GENERALIZACIÓN A UNA LEY NUNCA VISTA (Fung) Y CONTROL DE CLASE A LA ENTRADA
#   Fung: P = C + A·(exp(B·(λ − λ_min)) − 1), parámetros mezclados de los ajustes a sujetos de entrenamiento.
#   Control de clase: debe aceptar curvas J (Fung, sujetos) y rechazar sujetos de test modulados con una oscilación.
import numpy as np
from scipy.optimize import curve_fit

N_OOD, RUIDO_OOD = 300, 0.01
rng_ood = np.random.default_rng(2027)
fung = lambda x, a, b, c: c + a * (np.exp(b * (x - X_MIN)) - 1.0)
P_FUNG = np.array([curve_fit(fung, X, y, p0=(5.0, 20.0, max(float(y[0]), 0.01)), bounds=([1e-6, 1e-3, 0], [1e4, 200, 1e3]),
                             maxfev=20000)[0] for y in POB_Y[MASK_TRAIN]])
Y_fung = []
while len(Y_fung) < N_OOD:
    k = rng_ood.integers(2, 4)
    idx = rng_ood.choice(len(P_FUNG), size=k, replace=False)
    y = fung(X, *np.exp(rng_ood.dirichlet(np.ones(k)) @ np.log(P_FUNG[idx]) + rng_ood.normal(0.0, 0.10, size=3)))
    if y.min() > 0 and y.max() <= Y_TECHO:
        Y_fung.append(y)
Y_fung = np.array(Y_fung)
Y_fung_r = np.maximum(Y_fung + rng_ood.normal(0.0, RUIDO_OOD, size=Y_fung.shape) * np.abs(Y_fung), 0.0)
Mf_rn, Mf_opt = metricas(Y_fung_r, predecir(Y_fung_r) @ A.T), metricas(Y_fung_r, mco(Y_fung_r) @ A.T)
i = int(np.argmin(Mf_rn["R2"]))
print(f"🧪 Fung ({N_OOD} curvas, ruido 1 %, {np.mean((Y_fung < Y1) | (Y_fung > Y2)):.1%} de puntos fuera de y1–y2): "
      f"NRMSE red {Mf_rn['NRMSE'].mean():.3f}% | óptimo {Mf_opt['NRMSE'].mean():.3f}% | "
      f"exceso {(Mf_rn['NRMSE'].mean() / Mf_opt['NRMSE'].mean() - 1) * 100:+.1f}% | peor R² red {Mf_rn['R2'][i]:.5f} "
      f"(óptimo {Mf_opt['R2'][i]:.5f}) | MCO fabricable {np.mean((mco(Y_fung_r) >= 0).all(1)):.1%}")

t = X - X_MIN
Y_mod = []
for y in POB_Y[MASK_TEST]:
    for _ in range(max(1, N_OOD // (4 * MASK_TEST.sum()))):
        f, a = rng_ood.uniform(1.0, 3.0), rng_ood.uniform(0.10, 0.30)
        Y_mod.append(y * (1.0 + a * np.sin(2 * np.pi * f * t / t[-1] + rng_ood.uniform(0, 2 * np.pi))))
Y_mod = np.array(Y_mod)
Y_mod_r = np.maximum(Y_mod + rng_ood.normal(0.0, RUIDO_OOD, size=Y_mod.shape) * np.abs(Y_mod), 0.0)
no_J = np.array([not all(diagnostico_forma(y)) for y in Y_mod])
acepta = lambda Y: np.array([distancia_clase(y) <= TOL_CLASE_NRMSE for y in Y])
a_mod = acepta(Y_mod_r)
print(f"🛂 Control de clase (≤ {TOL_CLASE_NRMSE}%): acepta {acepta(Y_fung_r).mean():.1%} de las Fung y {acepta(POB_Y).mean():.0%} "
      f"de los sujetos | rechaza {(~a_mod[no_J]).mean():.1%} de las moduladas fuera de la clase (n = {no_J.sum()}); "
      f"acepta {a_mod[~no_J].mean():.1%} de las que siguieron siendo J (n = {(~no_J).sum()})")

## Celda 18 — Muestras aleatorias de test

In [ ]:
# CELDA 18: MUESTRAS ALEATORIAS DE TEST (2 por familia; escala vertical por panel)
import numpy as np
import matplotlib.pyplot as plt

rng_plot = np.random.default_rng(SEED)
muestras = [rng_plot.choice(np.where(FAM_T == f)[0], size=2, replace=False) for f in sorted(FAMILIAS)]
fig, axes = plt.subplots(2, len(FAMILIAS), figsize=(5.2 * len(FAMILIAS), 10))
for c, (f, ss) in enumerate(zip(sorted(FAMILIAS), muestras)):
    for r, s in enumerate(ss):
        panel(axes[r, c], s, f"#{s} — {FAMILIAS[f]}\n$R^2$ red {M_RN['R2'][s]:.5f} (óptimo {M_OPT['R2'][s]:.5f}) | "
                             f"MaxAE {M_RN['MaxAE'][s]:.3f} kPa")
plt.tight_layout()
plt.savefig(f"reconstruccion_ejemplos_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 19 — Diseñador de injerto

In [ ]:
# CELDA 19: DISEÑADOR — curva σθθ(λθ) → láminas de cada material
#   MODO = "masson": un sujeto ("HT07") o una curva media ("Mean HT") | "excel": columnas (λθ, σθθ [kPa]) | "puntos": PUNTOS
#   Resultado del método: MCO exacto (red + pulido si la curva es J; MCO directo si no). La red sola se muestra para comparar.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

MODO, SUJETO_EVAL = "masson", "Mean HT"
ARCHIVO_CURVA = "Curva_J.xlsx"
PUNTOS = [(1.02, 20.0), (1.07, 35.0), (1.10, 50.0), (1.13, 75.0), (1.16, 120.0)]   # (λθ, σθθ [kPa]), interpolación PCHIP

if MODO == "masson":
    y_t = MEDIAS[SUJETO_EVAL] if SUJETO_EVAL in MEDIAS else POB_Y[(POB.id == SUJETO_EVAL).to_numpy()][0]
else:
    xu, yu = (np.array(PUNTOS, float).T if MODO == "puntos" else pd.read_excel(ARCHIVO_CURVA, header=None).to_numpy(float)[:, :2].T)
    o = np.argsort(xu)
    y_t = PchipInterpolator(xu[o], yu[o], extrapolate=False)(X) if MODO == "puntos" else np.interp(X, xu[o], yu[o])
    y_t = np.where(np.isnan(y_t), np.interp(X, xu[o], yu[o]), y_t)
en_clase = distancia_clase(y_t) <= TOL_CLASE_NRMSE
c_opt, c_rn = mco(y_t), predecir(y_t)
m_opt, m_rn = metricas(y_t, A @ c_opt), metricas(y_t, A @ c_rn)
print(f"🫀 {SUJETO_EVAL if MODO == 'masson' else MODO} | curva J: {'sí' if en_clase else 'NO (resultado = MCO directo)'} | "
      f"dentro de y1–y2: {bool(dentro_espacio(y_t)[0])} | fabricable (c ≥ 0): {bool((c_opt >= 0).all())}")
print(pd.DataFrame({"MCO": c_opt, "red": c_rn, "diferencia": c_rn - c_opt}, index=pd.Index(range(1, len(NUDOS) + 1), name="material"))
      .round(5).to_string())
print(f"Total de láminas: MCO {c_opt.sum():.3f} | red {c_rn.sum():.3f}")
print(pd.DataFrame({"MCO": m_opt, "red": m_rn}).T.round(6).to_string())

DISENO = pd.DataFrame({"nudo λ": NUDOS, "holgura de fibra [%]": (NUDOS / NUDOS[0] - 1) * 100, "láminas c": c_opt,
                       "C_LAM": C_LAM, f"σ de 1 lámina en λ={X_MAX:g} [kPa]": A[-1], f"aporte σ λ={X_MAX:g} [kPa]": c_opt * A[-1],
                       "deform. máx. [%]": (X_MAX / NUDOS - 1) * 100}, index=pd.Index(range(1, len(NUDOS) + 1), name="material"))
print(f"\n🧵 σθθ en λ = {X_MAX:g}: {A[-1] @ c_opt:.2f} kPa | Δσ = {np.ptp(y_t):.2f} kPa")
print(DISENO.round(4).to_string())
DISENO.to_csv(f"diseno_injerto_{VERSION}.csv")

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
banda(ax[0])
for b, st, n in [(y_t, "k-", "Curva objetivo"), (A @ c_opt, "b:", "Óptimo MCO"), (A @ c_rn, "r--", "Red")]:
    ax[0].plot(X, b, st, lw=2, label=n)
    if n != "Curva objetivo":
        ax[1].plot(X, y_t - b, st, lw=1.8, label=n)
        ax[2].plot(X, np.abs(y_t - b) / np.ptp(y_t) * 100, st, lw=1.8, label=n)
for k_ in NUDOS[NUDOS >= X_MIN]:
    ax[1].axvline(k_, color="gray", ls="-.", alpha=0.5)
ax[2].axhline(PISO_MAXAE, color="gray", ls=":", label=f"Peor sujeto real ({PISO_MAXAE:.2f}%)")
for a, yl, ti in zip(ax, [ETQ_Y, "Residuo [kPa]", "Error [% del rango]"], [SUJETO_EVAL, "Residuos", "Error punto a punto"]):
    a.set(xlabel=ETQ_X, ylabel=yl, title=ti)
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"evaluacion_inferencia_curva_j_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 20 — Reproducción de láminas aisladas

In [ ]:
# CELDA 20: REPRODUCCIÓN DE LÁMINAS AISLADAS (entrada y = a·A_j; respuesta exacta c = a·e_j)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AMPLITUDES = [0.25, 0.5, 1.0, 2.0, 4.0]                              # láminas del diseño
nb = len(NUDOS)
C_EX = np.vstack([a * np.eye(nb)[j] for j in range(nb) for a in AMPLITUDES])
Y_B = C_EX @ A.T
C_B = predecir(Y_B)
j_act = np.repeat(np.arange(nb), len(AMPLITUDES))
act = C_B[np.arange(len(C_B)), j_act]
DF_B = pd.DataFrame({"material": j_act + 1, "láminas a": np.tile(AMPLITUDES, nb), "R2_red": metricas(Y_B, C_B @ A.T)["R2"],
                     "error c activo %": (act / C_EX.max(1) - 1) * 100,
                     "crosstalk máx.": np.abs(np.where(C_EX > 0, 0.0, C_B)).max(1)})
print(DF_B.round(5).to_string(index=False))
print(f"Peor |error| del coeficiente activo: {DF_B['error c activo %'].abs().max():.3f}% | crosstalk máx.: {DF_B['crosstalk máx.'].max():.5f} láminas")
DF_B.to_csv(f"test_reproduccion_bases_{VERSION}.csv", index=False)

fig, ax = plt.subplots(1, nb + 1, figsize=(4.4 * (nb + 1), 4.2))
for j in range(nb):
    d = DF_B[DF_B.material == j + 1]
    ax[0].plot(d["láminas a"], d["error c activo %"], "o-", label=f"Material {j+1}")
    m = j_act == j
    ax[j + 1].plot(X, Y_B[m].T, "k-", lw=1.6)
    ax[j + 1].plot(X, (C_B[m] @ A.T).T, "r--", lw=1.3)
    ax[j + 1].set(title=f"Material {j+1}: a = {AMPLITUDES} (negro exacta, rojo red)", xlabel=ETQ_X, ylabel=ETQ_Y)
ax[0].axhline(0, color="k", ls="--", alpha=0.6)
ax[0].set(xscale="log", xlabel="Láminas a", ylabel="Error del coeficiente activo [%]", title="Reproducción de láminas aisladas")
ax[0].set_xticks(AMPLITUDES, [f"{a:g}" for a in AMPLITUDES])
ax[0].minorticks_off()
ax[0].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
for a in ax[1:]:
    a.title.set_fontsize(8)
plt.tight_layout()
plt.savefig(f"test_reproduccion_bases_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 21 — Las bases con su fórmula

In [ ]:
# CELDA 21: LAS BASES CON SU FÓRMULA (continua: dominio de ajuste; punteada: extensión de la misma fórmula)
import numpy as np
import matplotlib.pyplot as plt

LAMBDA_GRAF = (0.80, 1.20)                                           # rango del gráfico (p. ej. (0.0, 1.2))
lam = np.linspace(*LAMBDA_GRAF, 2401)
Ab = forma_base(lam, NUDOS) * S
dentro = (lam >= X_MIN) & (lam <= X_MAX)
print(f"Fórmulas verificadas contra las bases: |diferencia| máx. = {np.abs(forma_base(X, NUDOS) * S - A).max():.1e} kPa")
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.axvspan(X_MIN, X_MAX, color="#e5e7eb", alpha=0.7, lw=0, label=f"Dominio de ajuste [{X_MIN}, {X_MAX}]")
for j, c in enumerate(plt.cm.viridis(np.linspace(0, 0.85, len(NUDOS)))):
    f = f"A_{j+1}(λ) = {S[j]:.2f}·max(0, λ − {NUDOS[j]:.4f})".replace(".", ",")
    if j in FORMA_ENSAYO:
        f = (f"A_{j+1}(λ) = ensayo (columna material_{j + 1} de {ARCHIVO_ENSAYOS})"
             + f" (escalado a {S[j] * (X_MAX - NUDOS[j]):.2f} kPa en λ = {X_MAX:g})".replace(".", ","))
    print("   " + f)
    ax.plot(lam, np.where(dentro, Ab[:, j], np.nan), color=c, lw=2.2, label=f)
    ax.plot(lam, np.where(dentro, np.nan, Ab[:, j]), color=c, lw=1.4, ls="--")
ax.set(xlim=LAMBDA_GRAF, xlabel=ETQ_X, ylabel="Tensión σθθ de una lámina [kPa]",
       title=f"Una lámina de cada material (C_LAM = {C_LAM.tolist()})")
ax.grid(alpha=0.5, ls=":")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig(f"bases_funcionales_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Celda 22 — Extrapolación fuera del dominio de ajuste

In [ ]:
# CELDA 22: EXTRAPOLACIÓN FUERA DEL DOMINIO DE AJUSTE — injerto (láminas ajustadas en [1,02; 1,16]) vs. arteria
#   Las láminas no se reajustan: se evalúa la misma receta en LAMBDA_EXT. Bajo el primer nudo el injerto no aporta
#   tensión (las capas no trabajan a compresión), mientras la ley de Masson puede dar σθθ < 0 (fuera de lo medido).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAMBDA_EXT = (0.80, 1.20)
CURVAS_EXT = ["NT08", "Mean HT", SUJETO_Y2]                       # sujetos o curvas medias
le = np.linspace(*LAMBDA_EXT, 801)
A_ext = forma_base(le, NUDOS) * S
fig, axes = plt.subplots(1, len(CURVAS_EXT), figsize=(6.2 * len(CURVAS_EXT), 5))
filas = {}
for ax, k in zip(axes, CURVAS_EXT):
    sub = POB[GRUPOS_MEDIAS[k]] if k in GRUPOS_MEDIAS else POB[POB.id == k]
    y_art = np.mean([curva_objetivo(le, r) for _, r in sub.iterrows()], axis=0)
    y_inj = A_ext @ mco(MEDIAS[k] if k in MEDIAS else POB_Y[(POB.id == k).to_numpy()][0])
    filas[k] = {f"λ={l:g}: arteria / injerto [kPa]": f"{np.interp(l, le, y_art):.1f} / {np.interp(l, le, y_inj):.1f}"
                for l in (0.80, 0.90, 0.95, 1.00, 1.18, 1.20) if LAMBDA_EXT[0] <= l <= LAMBDA_EXT[1]}
    ax.axvspan(X_MIN, X_MAX, color="#e5e7eb", alpha=0.7, lw=0, label="Dominio de ajuste")
    ax.plot(le, y_art, "k-", lw=2, label="Arteria (ley de Masson)")
    ax.plot(le, y_inj, "r--", lw=2, label="Injerto (láminas MCO)")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set(xlabel=ETQ_X, ylabel=ETQ_Y, title=k, ylim=(max(y_art.min(), -0.3 * y_art.max()), 1.05 * max(y_art.max(), y_inj.max())))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(f"extrapolacion_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()
print(pd.DataFrame(filas).to_string())

## Celda 23 — Las dos formas: σθθ [kPa] y (σ − σ_mín)/Δσ

In [ ]:
# CELDA 23: LAS DOS FORMAS DE LAS CURVAS — σθθ [kPa] y (σ − σ_mín)/Δσ (adimensional), MCO y red contra la original
#   Δσ = σ(X_MAX) − σ(X_MIN): cuánto sube la tensión de ESA curva en el rango de λ. Cada curva tiene su Δσ.
#   La forma adimensional va de 0 (λ = X_MIN) a 1 (λ = X_MAX) y compara solo la forma. El error en % del rango es el mismo en las dos.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def adimensional(y, ref):
    """(y − σ_mín)/Δσ con σ_mín y Δσ de la curva original ref."""
    return (np.asarray(y) - ref[0]) / (ref[-1] - ref[0])


TRES = {"y2 (la más rígida)": Y2, "Mean HT": MEDIAS["Mean HT"], "y1 (la más blanda)": Y1}
fig, axs = plt.subplots(2, 3, figsize=(17, 9))
_fil = []
for i, (tit, y) in enumerate(TRES.items()):
    c_m, c_r = mco(y), predecir(y)
    b_mco, b_red = A @ c_m, A @ c_r
    e_m, e_r = err_pct(y, b_mco)[0][0], err_pct(y, b_red)[0][0]
    for fila, f, yl in ((0, lambda v: v, ETQ_Y), (1, lambda v: adimensional(v, y), ETQ_YN)):
        ax = axs[fila, i]
        ax.plot(X, f(y), "k-", lw=2.6, label="Original")
        ax.plot(X, f(b_mco), "b:", lw=2, label=f"MCO (error {e_m:.2f} %)")
        ax.plot(X, f(b_red), "r--", lw=1.8, label=f"Red (error {e_r:.2f} %)")
        ax.set(xlabel=ETQ_X, ylabel=yl, title=tit + ("" if fila == 0 else " — adimensional"))
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="upper left")
    _fil.append({"curva": tit, "σ_mín [kPa]": y[0], "σ_máx [kPa]": y[-1], "Δσ [kPa]": y[-1] - y[0],
                 "adimensional en λ=1,10": float(np.interp(1.10, X, adimensional(y, y))), "error MCO %": e_m, "error red %": e_r,
                 **{f"láminas MCO {j + 1}": c_m[j] for j in range(N_BASES)}, **{f"láminas red {j + 1}": c_r[j] for j in range(N_BASES)}})
fig.suptitle(f"{N_BASES} materiales · arriba: σθθ [kPa]; abajo: (σ − σ_mín)/Δσ · error = máximo punto a punto, % del rango")
fig.tight_layout()
plt.savefig(f"dos_formas_y2_meanHT_y1_{VERSION}.png", dpi=200, bbox_inches="tight")
plt.show()
_t = pd.DataFrame(_fil).set_index("curva")
print(_t.iloc[:, :7].round(3).to_string())
print("\nLáminas de cada material (MCO | red):")
print(_t.iloc[:, 7:].T.round(3).to_string())

_N = (POB_Y - POB_Y[:, :1]) / np.ptp(POB_Y, axis=1, keepdims=True)
fig, axs = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, D, yl in ((axs[0], POB_Y, ETQ_Y), (axs[1], _N, ETQ_YN)):
    if D is POB_Y:
        banda(ax)
    for g, c in (("NT", "tab:blue"), ("HT", "tab:red")):
        m = (POB.grupo == g).to_numpy()
        for d in D[m]:
            ax.plot(X, d, color=c, lw=0.8, alpha=0.5)
        ax.plot(X, D[m].mean(0), color=c, lw=3, label=f"Media {g} (n = {m.sum()})")
    ax.set(xlabel=ETQ_X, ylabel=yl)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="upper left")
axs[0].set_title("41 sujetos: σθθ [kPa]")
axs[1].set_title("41 sujetos: (σ − σ_mín)/Δσ")
fig.tight_layout()
plt.savefig(f"sujetos_dos_formas_{VERSION}.png", dpi=200, bbox_inches="tight")
plt.show()
pd.DataFrame(_N.T, index=pd.Index(np.round(X, 6), name="lambda_theta"), columns=POB["id"]).to_csv(f"curvas_adimensionales_{VERSION}.csv")
print(f"Δσ de los 41 sujetos: mín {np.ptp(POB_Y, 1).min():.1f} kPa, mediana {np.median(np.ptp(POB_Y, 1)):.1f} kPa, "
      f"máx {np.ptp(POB_Y, 1).max():.1f} kPa | curvas adimensionales: curvas_adimensionales_{VERSION}.csv")

## Celda 24 — Redondeo a láminas enteras: Schnorr–Euchner y minimax (MILP, HiGHS)

### Métodos: redondeo a láminas enteras

**Por qué.** El injerto se fabrica con un número entero de láminas de cada material. El MCO y la red dan números reales (por ejemplo 12,85 láminas); hay que pasarlos a enteros z = (z₁, …, z_N) ≥ 0 perdiendo lo menos posible.

**Notación.** y(λ) es la curva objetivo σθθ en 500 valores de λ entre 1,02 y 1,16; A es la matriz de bases (columna j = tensión de una lámina del material j); c_MCO = argmin‖y − A c‖₂ (solución real). El error se expresa en % del rango de la curva, Δσ = σ(1,16) − σ(1,02).

**1. Redondeo simple.** z = round(c_red). No mira la curva: puede alejarse mucho (peor caso 7,4 % en los 41 sujetos).

**2. Redondeo por mínimos cuadrados enteros (Schnorr–Euchner).** Busca el entero que minimiza el error cuadrático medio contra la curva:
z_SE = argmin_{z ∈ ℤᴺ, z ≥ 0} ‖y − A z‖₂.
Por Pitágoras, ‖y − A z‖² = ‖y − A c_MCO‖² + (z − c_MCO)ᵀ AᵀA (z − c_MCO), así que basta buscar el entero más cercano a c_MCO en la métrica AᵀA. Con la factorización de Cholesky AᵀA = RᵀR el problema se recorre como un árbol (de la última lámina a la primera) y se podan las ramas cuyo error parcial ya supera el mejor encontrado (enumeración de Schnorr y Euchner, 1994; Agrell et al., 2002). Es exacto y tarda menos de 1 ms por curva.

**3. Redondeo minimax por programación lineal entera mixta (MILP, HiGHS).** Busca el entero que minimiza el **peor** error en cualquier λ:
min t  sujeto a  −t ≤ y(λᵢ) − (A z)(λᵢ) ≤ t  (i = 1 … 500),  z entero ≥ 0,  t ≥ 0.
Es lineal en (z, t), con N variables enteras y una real. Se resuelve con el método de ramificación y acotamiento del programa HiGHS (Huangfu y Hall, 2018) a través de `scipy.optimize.milp`, sin acotar la búsqueda: da el óptimo global. Tarda ≈ 70 ms por curva.

**Qué usa cada uno.** Ni Schnorr–Euchner (que solo necesita c_MCO, que a su vez sale de la curva) ni el minimax usan la red: dependen solo de la curva y de las bases. Cada uno gana en su propia medida: Schnorr–Euchner en error cuadrático medio (E_rms), el minimax en error máximo (e_máx). El criterio de fabricación es el error máximo (≤ 3 % en los sujetos, ≤ 4 % en el espacio de trabajo), por eso el método recomendado es el minimax.

**Ejemplo a mano (y2, 6 materiales).** MCO = (19,86; 10,33; 12,85; 20,00; 17,78; 14,58), e_máx = 0,94 %. Redondeo simple (20, 10, 13, 20, 18, 15): e_máx = 1,21 %. Minimax (20, 11, 11, 20, 19, 14): e_máx = 0,91 %. Frente al redondeo simple, el minimax quita dos láminas del material 3 y una del 6, y agrega una del 2 y una del 5: no es "redondear cada número", es buscar la mejor combinación.

**Referencias.** Schnorr, C. P. y Euchner, M. (1994), *Math. Programming* 66, 181–199. Agrell, E., Eriksson, T., Vardy, A. y Zeger, K. (2002), *IEEE Trans. Inf. Theory* 48(8), 2201–2214. Huangfu, Q. y Hall, J. A. J. (2018), *Math. Programming Computation* 10, 119–142. Virtanen, P. et al. (2020), SciPy 1.0, *Nature Methods* 17, 261–272.

In [ ]:
# CELDA 24: LÁMINAS ENTERAS — redondeo óptimo contra la curva de la arteria
#   «Redondeo minimax por programación lineal entera mixta (MILP, HiGHS)»: enteros z ≥ 0 de menor error punto a punto máximo,
#       min t  s.a.  −t ≤ y(λ_i) − Σ_j z_j·A_j(λ_i) ≤ t,  z_j entero ≥ 0   (HiGHS: ramificación y acotamiento con planos de corte).
#       Usa solo la curva y las bases (ni el MCO ni la red).
#   «Redondeo por mínimos cuadrados enteros (Schnorr–Euchner)»: enteros de menor E_rms contra la arteria. Como c_MCO es la proyección
#       de y sobre las bases, ||y − A·z||² = ||y − A·c_MCO||² + ||A·(z − c_MCO)||² (Pitágoras): se busca el entero más cercano a c_MCO
#       en la métrica AᵀA, por enumeración de Schnorr–Euchner sobre la factorización de Cholesky (búsqueda exacta).
#   «Redondeo simple»: cada lámina de la red al entero más cercano (referencia: no se recomienda).
import os
import sys
import time
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds

N_MUESTRA_ESPACIO = 2000                 # curvas del espacio de trabajo que se redondean (las 22 500 tardan ≈ 20 min)


@contextlib.contextmanager
def _silencio():
    sys.stdout.flush()
    fd, nul = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    os.dup2(nul, 1)
    try:
        yield
    finally:
        os.dup2(fd, 1); os.close(fd); os.close(nul)


def redondeo_minimax(y, B=None, z0=None, caja=None):
    """Redondeo minimax por MILP (HiGHS) en 500 puntos de λ. z0 y caja (± láminas alrededor de z0) solo acotan la búsqueda."""
    B = A if B is None else B
    Bs, ys = B[::2], np.asarray(y, float)[::2]
    m, n = Bs.shape
    lo, hi = np.zeros(n + 1), np.r_[np.full(n, np.inf), np.inf]
    if z0 is not None and caja is not None:
        lo[:n] = np.maximum(np.round(z0) - caja, 0)
        hi[:n] = np.round(z0) + caja
    with _silencio():
        r = milp(np.r_[np.zeros(n), 1.0], integrality=np.r_[np.ones(n), 0], bounds=Bounds(lo, hi),
                 constraints=LinearConstraint(np.vstack([np.c_[Bs, -np.ones(m)], np.c_[-Bs, -np.ones(m)]]), -np.inf, np.r_[ys, -ys]),
                 options=dict(time_limit=60))
    if r.x is None:                     # sin solución en 60 s (raro): se usa el de mínimos cuadrados enteros
        return redondeo_mce(np.linalg.lstsq(B, np.asarray(y, float), rcond=None)[0], B)
    return np.round(r.x[:n])


def redondeo_mce(c, B=None):
    """Redondeo por mínimos cuadrados enteros (Schnorr–Euchner): argmin_{z entero ≥ 0} (z − c)ᵀ BᵀB (z − c), con c = c_MCO."""
    B = A if B is None else B
    M = B.T @ B
    R = np.linalg.cholesky(M).T
    z0 = np.maximum(np.round(c), 0)
    mejor = [float((z0 - c) @ M @ (z0 - c)), z0.copy()]
    z = np.zeros(len(c))

    def nivel(k, parcial):
        s = c[k] - (R[k, k + 1:] @ (z[k + 1:] - c[k + 1:])) / R[k, k]
        radio = np.sqrt(max(mejor[0] - parcial, 0.0)) / abs(R[k, k])
        for v in sorted(range(max(0, int(np.ceil(s - radio))), int(np.floor(s + radio)) + 1), key=lambda v: abs(v - s)):
            z[k] = v
            p = parcial + (R[k, k] * (v - s)) ** 2
            if p < mejor[0]:
                if k == 0:
                    mejor[0], mejor[1] = p, z.copy()
                else:
                    nivel(k - 1, p)
    nivel(len(c) - 1, 0.0)
    return mejor[1]


def errores_curva(y, z, B=None):
    """(E_rms, e_máx) en % del rango de la curva y."""
    B = A if B is None else B
    r = np.asarray(y) - B @ z
    return 100 * np.sqrt(np.mean(r ** 2)) / np.ptp(y), 100 * np.abs(r).max() / np.ptp(y)


METODOS = ["MCO (sin redondear)", "Red neuronal (sin redondear)", "Redondeo simple (de la red)",
           "Mínimos cuadrados enteros (Schnorr–Euchner)", "Minimax (MILP, HiGHS)"]


def disenos_enteros(y):
    cm, cr = mco(y), predecir(y)
    return dict(zip(METODOS, [cm, cr, np.maximum(np.round(cr), 0), redondeo_mce(cm), redondeo_minimax(y)]))


# --- y2, Mean HT, y1: tablas y curvas ---
TRES = {"y2": Y2, "Mean HT": MEDIAS["Mean HT"], "y1": Y1}
DISENOS_3 = {nom: disenos_enteros(y) for nom, y in TRES.items()}
for nom, y in TRES.items():
    t = pd.DataFrame({m: {**{f"c{j + 1}": z[j] for j in range(N_BASES)},
                          **dict(zip(["E_rms %", "e_máx %"], errores_curva(y, z)))} for m, z in DISENOS_3[nom].items()}).T
    print(f"\n{nom}: láminas de cada material (C_LAM = {int(C_LAM[0])}) y errores contra la curva original [% del rango]")
    print(t.to_string(float_format=lambda v: f"{v:.2f}" if abs(v - round(v)) > 1e-9 else f"{v:.0f}"))
_EST = {"MCO (sin redondear)": ("tab:blue", ":", 1.8), "Red neuronal (sin redondear)": ("tab:red", "--", 1.4),
        "Redondeo simple (de la red)": ("tab:gray", "-.", 1.4), "Mínimos cuadrados enteros (Schnorr–Euchner)": ("tab:orange", "--", 1.8),
        "Minimax (MILP, HiGHS)": ("tab:green", "-", 2.2)}
fig, axs = plt.subplots(3, 3, figsize=(18, 13), gridspec_kw=dict(height_ratios=[1.4, 1.1, 1]))
for i, (nom, y) in enumerate(TRES.items()):
    ax, axn, axr = axs[0, i], axs[1, i], axs[2, i]
    ax.plot(X, y, "k-", lw=2.8, label="Original")
    axn.plot(X, (y - y[0]) / np.ptp(y), "k-", lw=2.8)
    for m, z in DISENOS_3[nom].items():
        col, ls, lw = _EST[m]
        p = A @ z
        e = errores_curva(y, z)
        ax.plot(X, p, color=col, ls=ls, lw=lw, label=f"{m} (e_máx {e[1]:.2f} %)")
        axn.plot(X, (p - y[0]) / np.ptp(y), color=col, ls=ls, lw=lw)
        axr.plot(X, (p - y) / np.ptp(y) * 100, color=col, ls=ls, lw=lw)
    axr.axhline(0, color="k", lw=0.8)
    ax.set(title=f"{nom}: σθθ [kPa]", xlabel=ETQ_X, ylabel=ETQ_Y)
    axn.set(title=f"{nom}: (σ − σ_mín)/Δσ", xlabel=ETQ_X, ylabel=ETQ_YN)
    axr.set(title=f"{nom}: diseño − original [% del rango]", xlabel=ETQ_X, ylabel="%")
    ax.legend(fontsize=7.5, loc="upper left")
    for a in (ax, axn, axr):
        a.grid(alpha=0.3)
fig.suptitle(f"{N_BASES} materiales · láminas reales y enteras contra la curva original (C_LAM = {int(C_LAM[0])})")
fig.tight_layout()
plt.savefig(f"redondeo_y2_meanHT_y1_{VERSION}.png", dpi=200, bbox_inches="tight")
plt.show()

# --- 41 sujetos y muestra del espacio de trabajo ---
_t0 = time.time()
_fil = []
_idx = np.random.default_rng(SEED).choice(len(W_ESP), min(N_MUESTRA_ESPACIO, len(W_ESP)), replace=False)
for grupo, Ys, noms in (("41 sujetos", POB_Y, list(POB.id)), (f"{len(_idx)} curvas del espacio", W_ESP[_idx], [f"W{i}" for i in _idx])):
    for y, nm in zip(Ys, noms):
        for m, z in disenos_enteros(y).items():
            e = errores_curva(y, z)
            _fil.append({"grupo": grupo, "curva": nm, "método": m, "E_rms %": e[0], "e_máx %": e[1]})
REDONDEO = pd.DataFrame(_fil)
REDONDEO.to_csv(f"redondeo_{VERSION}.csv", index=False)
_res = REDONDEO.groupby(["grupo", "método"], sort=False).agg(
    E_rms_mediana=("E_rms %", "median"), e_máx_mediana=("e_máx %", "median"), e_máx_peor=("e_máx %", "max"),
    cumple_3=("e_máx %", lambda v: f"{(v <= 3).sum()}/{len(v)}"), cumple_4=("e_máx %", lambda v: f"{(v <= 4).sum()}/{len(v)}"))
print(f"\nResumen (errores contra la curva original, % del rango) | {time.time() - _t0:.0f} s:")
print(_res.round(3).to_string())
_g = REDONDEO[REDONDEO.método.isin(METODOS[3:])].pivot_table(index=["grupo", "curva"], columns="método", values=["E_rms %", "e_máx %"])
print("\nCada método gana en su medida: Schnorr–Euchner tiene menor o igual E_rms en "
      f"{(_g[('E_rms %', METODOS[3])] <= _g[('E_rms %', METODOS[4])] + 1e-9).mean():.1%} de las curvas; el minimax, menor o igual e_máx en "
      f"{(_g[('e_máx %', METODOS[4])] <= _g[('e_máx %', METODOS[3])] + 1e-9).mean():.1%}.")

## Celda 25 — Robustez: ruido y pocos puntos

In [ ]:
# CELDA 25: ¿CUÁNDO APORTA LA RED? — prueba de robustez con curvas imperfectas (ruido y pocos puntos)
#   A cada sujeto se le agrega ruido (proporcional al valor) y/o se lo «mide» en pocos puntos (interpolación PCHIP entre ellos).
#   El minimax recibe tres curvas distintas: (1) la medida, (2) la reconstruida por el MCO (A·c_MCO), (3) la reconstruida por la red
#   (A·c_red). El diseño entero se compara con la curva VERDADERA (sin ruido). El minimax no depende del punto de partida: lo que
#   cambia es qué curva se le pide copiar.
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator

ESCENARIOS = {"ruido 1 %": (0.01, None), "ruido 3 %": (0.03, None), "10 puntos": (0.0, 10), "10 puntos + ruido 2 %": (0.02, 10)}
REPETICIONES = 5                         # realizaciones de ruido por sujeto y escenario
_rng = np.random.default_rng(SEED)
_fil = []
for esc, (ruido, npts) in ESCENARIOS.items():
    for rep in range(REPETICIONES):
        for y, nm in zip(POB_Y, POB.id):
            if npts:
                xm = np.linspace(X_MIN, X_MAX, npts)
                ym = np.interp(xm, X, y)
            else:
                xm, ym = X, y.copy()
            ym = ym + _rng.normal(0.0, ruido, size=ym.shape) * np.abs(ym)
            y_med = PchipInterpolator(xm, ym)(X) if npts else ym
            cm, cr = mco(y_med), predecir(y_med)
            curvas = {"(1) curva medida": y_med, "(2) curva del MCO": A @ cm, "(3) curva de la red": A @ cr}
            for k, yc in curvas.items():
                z = redondeo_minimax(yc)
                _fil.append({"escenario": esc, "curva al minimax": k, "sujeto": nm, "e_máx %": errores_curva(y, z)[1]})
            for k, c in (("continuo: MCO", cm), ("continuo: red", cr)):
                _fil.append({"escenario": esc, "curva al minimax": k, "sujeto": nm, "e_máx %": errores_curva(y, c)[1]})
ROBUSTEZ = pd.DataFrame(_fil)
ROBUSTEZ.to_csv(f"robustez_{VERSION}.csv", index=False)
print(f"Error del diseño contra la curva VERDADERA, e_máx [% del rango] | 41 sujetos × {REPETICIONES} realizaciones:")
print(ROBUSTEZ.groupby(["escenario", "curva al minimax"], sort=False)["e_máx %"].agg(
    mediana="median", p90=lambda v: np.percentile(v, 90), peor="max", cumple_4=lambda v: f"{(v <= 4).mean():.0%}").round(2).to_string())

## Celda 26 — Tiempo de cálculo por curva

In [ ]:
# CELDA 26: TIEMPO POR CURVA — red, MCO, minimax (MILP, HiGHS) y mínimos cuadrados enteros (Schnorr–Euchner)
import time
import numpy as np
import pandas as pd

_Y = POB_Y
_t = {}
t0 = time.perf_counter(); _cr = predecir(_Y); _t["Red (41 curvas juntas)"] = (time.perf_counter() - t0) / len(_Y)
t0 = time.perf_counter(); [predecir(y) for y in _Y[:10]]; _t["Red (de a una curva)"] = (time.perf_counter() - t0) / 10
t0 = time.perf_counter(); _cm = mco(_Y); _t["MCO (41 curvas juntas)"] = (time.perf_counter() - t0) / len(_Y)
t0 = time.perf_counter(); [redondeo_mce(c) for c in _cm]; _t["Mínimos cuadrados enteros (Schnorr–Euchner)"] = (time.perf_counter() - t0) / len(_Y)
t0 = time.perf_counter(); [redondeo_minimax(y) for y in _Y]; _t["Minimax (MILP, HiGHS)"] = (time.perf_counter() - t0) / len(_Y)
t0 = time.perf_counter(); [redondeo_minimax(y, z0=c, caja=3) for y, c in zip(_Y, _cr)]; _t["Minimax con caja ±3 alrededor de la red"] = (time.perf_counter() - t0) / len(_Y)
_ig = np.mean([np.array_equal(redondeo_minimax(y), redondeo_minimax(y, z0=c, caja=3)) for y, c in zip(_Y, _cr)])
print(pd.Series({k: f"{1e3 * v:.2f} ms" for k, v in _t.items()}, name="tiempo por curva").to_string())
print(f"Minimax con caja ±3 alrededor de la red: mismo diseño entero que sin caja en {_ig:.0%} de los sujetos")

## Celda 27 — Comparación: red entrenada con enteros (V40) vs red entrenada con MCO real (V39)

In [ ]:
# CELDA 27: COMPARACIÓN DE LAS DOS REDES — entrenada con láminas reales del MCO (V39) vs. con láminas enteras minimax (V40)
#   Para cada red: (a) su salida tal cual (real), (b) redondeo simple de su salida, (c) redondeo óptimo de su salida:
#   mínimos cuadrados enteros (Schnorr–Euchner) y minimax (MILP, HiGHS) sobre la curva de la red (A·c_red).
#   Referencia: minimax sobre la curva original (el óptimo entero, que no depende de ninguna red).
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

_m2 = load_model(f"modelo_rn_curva_j_{MODELO_COMPARACION}.keras", compile=False)
with open(f"scalers_curva_j_{MODELO_COMPARACION}.pkl", "rb") as f:
    _meta2 = pickle.load(f)
REDES = {"Red MCO (V39)": (_m2, _meta2["scaler_x"], _meta2["scaler_y"]), "Red enteras (V40)": (model, sx, sy)}


def predecir_con(red, Y):
    m, sx_, sy_ = REDES[red]
    c = sy_.inverse_transform(m.predict(sx_.transform(entrada_log(np.atleast_2d(Y))), batch_size=1024, verbose=0)) * ESC
    return c[0] if np.ndim(Y) == 1 else c


def variantes(y):
    out = {"Minimax sobre la curva original (referencia)": redondeo_minimax(y)}
    for red in REDES:
        c = predecir_con(red, y)
        out[f"{red}: salida real"] = c
        out[f"{red}: redondeo simple"] = np.maximum(np.round(c), 0)
        out[f"{red}: Schnorr–Euchner sobre su curva"] = redondeo_mce(c)
        out[f"{red}: minimax sobre su curva"] = redondeo_minimax(A @ c)
    return out


_fil = []
_idx = np.random.default_rng(SEED + 1).choice(len(W_ESP), min(N_MUESTRA_ESPACIO, len(W_ESP)), replace=False)
CURVAS27 = {**dict(zip(POB.id, POB_Y)), "y2": Y2, "Mean HT": MEDIAS["Mean HT"], "y1": Y1, **{f"W{i}": W_ESP[i] for i in _idx}}
DET27 = {}
for nom, y in CURVAS27.items():
    V = variantes(y)
    if nom in ("y2", "Mean HT", "y1"):
        DET27[nom] = V
    grupo = "41 sujetos" if nom in set(POB.id) else ("y2, Mean HT, y1" if nom in ("y2", "Mean HT", "y1") else f"{len(_idx)} curvas del espacio")
    for k, z in V.items():
        e = errores_curva(y, z)
        _fil.append({"grupo": grupo, "curva": nom, "variante": k, "E_rms %": e[0], "e_máx %": e[1]})
COMP27 = pd.DataFrame(_fil)
COMP27.to_csv(f"comparacion_redes_{VERSION}.csv", index=False)
print("Errores contra la curva original [% del rango]:")
print(COMP27.groupby(["grupo", "variante"], sort=False).agg(
    E_rms_mediana=("E_rms %", "median"), e_máx_mediana=("e_máx %", "median"), e_máx_peor=("e_máx %", "max"),
    cumple_3=("e_máx %", lambda v: f"{(v <= 3).sum()}/{len(v)}"), cumple_4=("e_máx %", lambda v: f"{(v <= 4).sum()}/{len(v)}")).round(3).to_string())
for nom in DET27:
    print(f"\n{nom}: láminas")
    print(pd.DataFrame({k: z for k, z in DET27[nom].items()}, index=[f"c{j + 1}" for j in range(N_BASES)]).T.round(2).to_string())

fig, axs = plt.subplots(2, 3, figsize=(18, 9), gridspec_kw=dict(height_ratios=[1.5, 1]))
_sel = {"Minimax sobre la curva original (referencia)": ("k", ":", 2.2), "Red MCO (V39): salida real": ("tab:blue", "--", 1.4),
        "Red enteras (V40): salida real": ("tab:red", "--", 1.4), "Red MCO (V39): redondeo simple": ("tab:cyan", "-.", 1.4),
        "Red enteras (V40): redondeo simple": ("tab:orange", "-.", 1.4)}
for i, (nom, V) in enumerate(DET27.items()):
    y = CURVAS27[nom]
    ax, axr = axs[0, i], axs[1, i]
    ax.plot(X, y, color="gray", lw=4, alpha=0.5, label="Original")
    for k, (col, ls, lw) in _sel.items():
        p = A @ V[k]
        ax.plot(X, p, color=col, ls=ls, lw=lw, label=f"{k} (e_máx {errores_curva(y, V[k])[1]:.2f} %)")
        axr.plot(X, (p - y) / np.ptp(y) * 100, color=col, ls=ls, lw=lw)
    axr.axhline(0, color="k", lw=0.8)
    ax.set(title=nom, xlabel=ETQ_X, ylabel=ETQ_Y)
    axr.set(xlabel=ETQ_X, ylabel="diseño − original [% del rango]")
    ax.legend(fontsize=7, loc="upper left")
    for a in (ax, axr):
        a.grid(alpha=0.3)
fig.suptitle(f"{N_BASES} materiales · red entrenada con láminas reales (V39) vs. con láminas enteras (V40)")
fig.tight_layout()
plt.savefig(f"comparacion_redes_{VERSION}.png", dpi=200, bbox_inches="tight")
plt.show()